# MB8 · Gradient de politique — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/mb8_gradient_politique/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/mb8_gradient_politique/03_notebook.ipynb`, créée par `python tools/start_chapter.py MB8`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | MB8.1 à MB8.7, MB8.11 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | MB8.13 | L'estimateur score d'une dérivée sur un échantillon, avec une baseline constante ou la moyenne des autres tirages, et son erreur type · `score_function_gradient` | 🔨 | ★ | 25 |
|  | MB8.14 | Le gradient exact de la valeur d'une politique softmax tabulaire, par le système de Bellman · `policy_gradient_exact` | 🔨 | ★★ | 35 |
|  | MB8.15 | Des poids d'importance calculés en log, auto-normalisés ou tronqués, et la taille effective de l'échantillon · `importance_weights` | 🔨 | ★★ | 25 |
|  | MB8.16 | Les forces d'un modèle de Bradley-Terry par maximum de vraisemblance (l'algorithme de Zermelo), centrées, et le refus d'un tournoi où elles n'existent pas · `bradley_terry_fit` | 🔨 | ★★ | 35 |
| B | MB8.17 | Prédire l'ordre des écarts-types de l'estimateur score d'une même dérivée, sans baseline, avec la moyenne des autres tirages et avec la baseline optimale, puis les mesurer | 🔮 | ★ | 20 |
|  | MB8.18 | Un événement rare, P(Z > 4) pour Z normale centrée réduite : Monte-Carlo naïf contre l'échantillonnage d'importance sous N(4, 1), la taille effective, puis des poids tronqués à c : le biais et la variance selon c | 🔬 | ★★ | 30 |
| C | MB8.19 | Chasse à l'erreur : le score d'une loi exponentielle écrit pour la moyenne au lieu du taux, des poids d'importance inversés (q/p), une baseline qui contient le tirage lui-même, des forces de deux tournois sans joueur commun comparées sur une même échelle | 🐛 | ★ | 25 |
|  | MB8.20 | Un test de propriété pour le gradient exact : sur des MDP tirés au hasard, comparé à des différences finies centrées, et de somme nulle sur les actions de chaque état | 🛠️ | ★ | 25 |
| D | MB8.21 | Défi : estimer, avec un budget de tirages, la dérivée d'une probabilité de dépassement par rapport au paramètre d'une loi, assez précisément neuf fois sur dix (score, baseline, variable de contrôle) | 🏆 | ★★★ | 40 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB8")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : une attente au téléphone, des graines en godet, un camion de crêpes, l'état d'une route, un panel de sondage, un tournoi de babyfoot, une cantine, et un pont vers la loss de substitution.
- Écrire les quatre fonctions de `mymath/policy_gradient.py` (`score_function_gradient`, `policy_gradient_exact`, `importance_weights`, `bradley_terry_fit`) et les tester.
- Prédire puis mesurer la dispersion de l'estimateur score pour trois baselines, estimer un événement rare par échantillonnage d'importance, corriger quatre erreurs, écrire deux tests de propriété, et relever un défi.

**Rappel express.** Le score en $\theta$ : $s_\theta(x) = \partial_\theta\ln p_\theta(x)$, d'espérance nulle quand le support ne dépend pas de $\theta$ ; $\frac{\mathrm{d}}{\mathrm{d}\theta}\mathbb{E}_\theta[f(X)] = \mathbb{E}_\theta[f(X)\,s_\theta(X)]$, et toute constante $b$ garde $(f - b)\,s$ sans biais ; $b^\star = \frac{\mathbb{E}[fs^2]}{\mathbb{E}[s^2]}$. Une politique softmax tabulaire : $\frac{\partial V(s_0)}{\partial z(s, a)} = \mathbf{M}[s_0, s]\,\pi(a \mid s)\,A(s, a)$, avec $\mathbf{M} = (\mathbf{I} - \gamma\mathbf{P}_\pi)^{-1}$. L'échantillonnage d'importance : $\mathbb{E}_p[f] = \mathbb{E}_q\big[\frac pq f\big]$, la taille effective $\frac{(\sum w_i)^2}{\sum w_i^2}$. Bradley-Terry : $P(i \text{ bat } j) = \sigma(\beta_i - \beta_j)$, victoires observées = victoires attendues au maximum.

## Partie 0 · Vérifier tes exercices papier (✏️ MB8.1 à MB8.7 ; 🔗 MB8.11)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]`, dans l'ordre que dit la question ; une matrice comme la liste de ses lignes, `[[1, 0], [0.5, 0.5]]`.

Les réponses pas encore remplies affichent ⏳.

**Ex MB8.1 — Une attente au téléphone de loi exponentielle de taux θ, et une pénalité au-delà de c minutes : d/dθ P(X > c) par la formule fermée, par l'estimateur score (son espérance calculée) et par le chemin X = ξ/θ ; la variance de l'estimateur score à un tirage**

In [ ]:
# Ex MB8.1: the values you found on paper
answer_MB8_1a = ...  # a) a number (4 decimals): P(X > c)
answer_MB8_1b = ...  # b) a number (4 decimals): d/dθ P(X > c), by the closed form
answer_MB8_1c = ...  # c) a list of 2 numbers (4 decimals): E[1{X > c}] / θ, then E[X 1{X > c}]
answer_MB8_1d = ...  # d) a list of 2 numbers (2 decimals): the path derivatives of E[X], then of P(X > c)
answer_MB8_1e = ...  # e) a number (2 decimals): the variance of g(X), at one draw

for letter, answer in zip("abcde", [answer_MB8_1a, answer_MB8_1b, answer_MB8_1c, answer_MB8_1d, answer_MB8_1e]):
    wbm.check(f"MB8.1{letter}", answer)

**Ex MB8.2 — Deux graines par godet (une loi binomiale B(2, θ)) et un prix selon le nombre de pousses : la dérivée de la recette moyenne, puis l'estimateur score sans baseline, avec b = E[f] et avec la baseline optimale, et leurs variances**

In [ ]:
# Ex MB8.2: the values you found on paper
answer_MB8_2a = ...  # a) a list of 3 numbers (2 decimals): the scores s(0), s(1), s(2)
answer_MB8_2b = ...  # b) a number (2 decimals): d/dθ E[f(X)]
answer_MB8_2c = ...  # c) a number (2 decimals): the variance without baseline (b = 0), at one draw
answer_MB8_2d = ...  # d) a number (2 decimals): the variance with b = E[f(X)], at one draw
answer_MB8_2e = ...  # e) a list of 2 numbers (3 decimals): b*, then the variance with b*

for letter, answer in zip("abcde", [answer_MB8_2a, answer_MB8_2b, answer_MB8_2c, answer_MB8_2d, answer_MB8_2e]):
    wbm.check(f"MB8.2{letter}", answer)

**Ex MB8.3 — Un camion de crêpes entre deux emplacements, une politique softmax dans l'un d'eux : la chaîne induite, les valeurs, la loi d'occupation actualisée, les valeurs et les avantages des deux actions, puis le gradient de la valeur par rapport aux deux logits**

In [ ]:
# Ex MB8.3: the values you found on paper
answer_MB8_3a = ...  # a) a 2 x 2 matrix (3 decimals), as a list of rows: P_pi (market, park)
answer_MB8_3b = ...  # b) a list of 2 numbers (2 decimals): V(market), V(park)
answer_MB8_3c = ...  # c) a list of 2 numbers (3 decimals): M[market, market], then M[park, market]
answer_MB8_3d = ...  # d) a list of 4 numbers (4 decimals): Q(market, stay), Q(market, leave), A(market, stay), A(market, leave)
answer_MB8_3e = ...  # e) a list of 4 numbers (4 decimals): dV(market)/dz(market, stay), dV(market)/dz(market, leave), dV(park)/dz(market, stay), dV(park)/dz(market, leave)

for letter, answer in zip("abcde", [answer_MB8_3a, answer_MB8_3b, answer_MB8_3c, answer_MB8_3d, answer_MB8_3e]):
    wbm.check(f"MB8.3{letter}", answer)

**Ex MB8.4 — L'état d'une route (bonne, usée, en travaux) et des valeurs estimées fausses : le biais des retours à 1 et 2 pas, les poids (1 − λ)λ^{k−1} d'un λ-retour, et son biais quand l'erreur est la même partout**

In [ ]:
# Ex MB8.4: the values you found on paper
answer_MB8_4a = ...  # a) a list of 3 numbers (3 decimals): the bias of the 1-step return from each state (good, worn, works)
answer_MB8_4b = ...  # b) a list of 3 numbers (4 decimals): the bias of the 2-step return from each state
answer_MB8_4c = ...  # c) a list of 2 numbers: the weight of the 5-step return (4 decimals), then an integer k
answer_MB8_4d = ...  # d) a number (4 decimals): the bias of the lambda-return when the error is 2 everywhere

for letter, answer in zip("abcd", [answer_MB8_4a, answer_MB8_4b, answer_MB8_4c, answer_MB8_4d]):
    wbm.check(f"MB8.4{letter}", answer)

**Ex MB8.5 — Un panel en ligne dont les classes d'âge ne suivent pas le recensement : les poids p/q, la dépense moyenne repondérée, la taille effective de Kish d'un petit échantillon, des poids tronqués (le biais), puis KL(q ‖ p) exacte et ses estimateurs k₁ et k₃ classe par classe**

In [ ]:
# Ex MB8.5: the values you found on paper
answer_MB8_5a = ...  # a) a list of 3 numbers (2 decimals): the weights w = p/q of the three classes
answer_MB8_5b = ...  # b) a list of 2 numbers (2 decimals): the raw mean spending E_q[f] (under q, not over the six panelists), then the reweighted one E_q[wf]
answer_MB8_5c = ...  # c) a number (2 decimals): Kish's effective size of the six panelists
answer_MB8_5d = ...  # d) a list of 2 numbers (2 decimals): the reweighted mean with the weights min(w, 1.6), then its bias
answer_MB8_5e = ...  # e) a number (5 decimals): KL(q || p)
answer_MB8_5f = ...  # f) a list of 3 numbers (4 decimals): k1 for a panelist of each class
answer_MB8_5g = ...  # g) a list of 3 numbers (4 decimals): k3 for a panelist of each class

for letter, answer in zip("abcdefg", [answer_MB8_5a, answer_MB8_5b, answer_MB8_5c, answer_MB8_5d, answer_MB8_5e, answer_MB8_5f, answer_MB8_5g]):
    wbm.check(f"MB8.5{letter}", answer)

**Ex MB8.6 — Un tournoi de babyfoot entre trois clubs : deux clubs seuls (la différence de forces en forme fermée), puis vérifier les équations du maximum de vraisemblance pour des forces proposées, la probabilité qu'un club batte un autre, une mise à jour Elo, et un club invaincu**

In [ ]:
# Ex MB8.6: the values you found on paper
answer_MB8_6a = ...  # a) a number (4 decimals): A and B alone, beta_A - beta_B
answer_MB8_6b = ...  # b) a list of 3 numbers (2 decimals): the expected wins of A against B, of A against C, of B against C
answer_MB8_6c = ...  # c) a list of 3 numbers (4 decimals): the centred strengths beta_A, beta_B, beta_C
answer_MB8_6d = ...  # d) a number (4 decimals): the probability that C beats A at a next match
answer_MB8_6e = ...  # e) a list of 2 numbers (1 decimal): the Elo gap R_A - R_C, then the points C wins if it beats A

for letter, answer in zip("abcde", [answer_MB8_6a, answer_MB8_6b, answer_MB8_6c, answer_MB8_6d, answer_MB8_6e]):
    wbm.check(f"MB8.6{letter}", answer)

**Ex MB8.7 — Trois plats à la cantine : la loi de service qui maximise la satisfaction moyenne moins β fois la KL à la loi actuelle, pour deux valeurs de β, sa valeur β ln Z, comparée à la loi actuelle et au meilleur plat seul**

In [ ]:
# Ex MB8.7: the values you found on paper
answer_MB8_7a = ...  # a) a list of 3 numbers (4 decimals): beta = 1, the law pi* (gratin, couscous, lasagne)
answer_MB8_7b = ...  # b) a number (4 decimals): beta = 1, the value beta ln Z
answer_MB8_7c = ...  # c) a list of 2 numbers (4 decimals): beta = 1, F(pi_ref), then F for the best dish alone
answer_MB8_7d = ...  # d) a list of 4 numbers (4 decimals): beta = 3, the law pi*, then the value beta ln Z

for letter, answer in zip("abcd", [answer_MB8_7a, answer_MB8_7b, answer_MB8_7c, answer_MB8_7d]):
    wbm.check(f"MB8.7{letter}", answer)

**Ex MB8.11 — Pont : la « loss de substitution » −(1/n)Σ(f(x_i) − b) ln p_θ(x_i), dont la dérivée, les f(x_i) et b tenus fixes, est moins l'estimateur score : sur le taux d'une loi de Poisson, puis ce que changent des f standardisés dans le batch**

In [ ]:
# Ex MB8.11: the values you found on paper
answer_MB8_11a = ...  # a) a list of 8 numbers (2 decimals): the eight scores at theta = 2.5
answer_MB8_11b = ...  # b) a number (4 decimals): dL/dtheta at theta = 2.5, with b = 1
answer_MB8_11c = ...  # c) a number (4 decimals): the same derivative, with b = 0

for letter, answer in zip("abc", [answer_MB8_11a, answer_MB8_11b, answer_MB8_11c]):
    wbm.check(f"MB8.11{letter}", answer)

## Partie A · Ta librairie : l'estimateur score, le gradient exact d'une valeur, les poids d'importance, les forces de Bradley-Terry

Fiche §2, §3, §4, §6 et §7. Les quatre fonctions de `mymath/policy_gradient.py`, leurs tests, et un usage de chacune. La cellule de données ci-dessous recopie les données des exercices papier.

In [ ]:
# Tools of the notebook (parts A to D)
import contextlib
import dis
import math
import numbers
import os
import re
import sys
import time
from fractions import Fraction
from types import MappingProxyType

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, refuses, run_pytest_group, verdict

TEST_MB8 = "tests/test_mb8_policy_gradient.py"       # the tests of mymath/policy_gradient.py
REF = wbm.load_mymath("ref", alias="mymath_ref")       # the reference: the colleague's code of MB8.19 relies on it
ref_pg = REF.policy_gradient


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())[:160]
    except (TypeError, ValueError):
        return repr(value)[:160]


GENERIC_COMPUTED = "Ta fonction, ou le calcul qui l'utilise, ne donne pas la bonne valeur ici."


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) or (
        isinstance(v, (list, tuple)) and any(isinstance(w, (bool, np.bool_)) for w in v)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown") and "Piste" not in message:
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            if message.startswith(GENERIC_COMPUTED):            # one text, not two: the hint says what to look at
                message = GENERIC_COMPUTED
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def library_hint(names):
    """A hint for a wrong answer computed with the learner's library (no classic error recognised): its tests first,
    then the call."""
    return lambda value: (f"si tu t'es servi de {names} de ta librairie et que ses tests passent tous (plus haut), "
                          "l'erreur est dans l'appel (les arguments, leur ordre, ce que tu lis dans le résultat) ; et "
                          "donne la valeur que rend le calcul, sans l'arrondir")


def passes(ex_id, value):
    """True when value is the hidden answer ex_id (wbm.check, computed=True, nothing printed)."""
    try:
        return bool(wbm.check(ex_id, value, computed=True, quiet=True).passed)
    except Exception:  # noqa: BLE001 - an odd value is not the answer
        return False


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def as_floats(value):
    """value as a float array, or None when it is not an array of numbers."""
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def call_safely(label, function, *args, **kwargs):
    """function(*args, **kwargs); an error of the learner's code becomes one ❌ line naming the call (StopCheck)."""
    try:
        return function(*args, **kwargs)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the call
        print(f"❌ {label} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        raise StopCheck from None


def frozen(array):
    """A float copy of array that refuses any change in place (the data given to the learner's functions)."""
    out = np.array(array, dtype=float)
    out.setflags(write=False)
    return out


class CodeFrom(BaseException):
    """Raised inside code_from when watched code starts (a BaseException: a broad except of the learner's code does
    not hide it)."""


_NOT_KEYED = {"CACHE", "PUSH_NULL", "RESUME", "NOP", "EXTENDED_ARG", "NOT_TAKEN"}


def code_key(code):
    """What identifies a compiled function whatever its file and the module around it: its instructions (without jump
    targets, nor what depends on the rest of the module) and the names of its variables."""
    ops = []
    for i in dis.get_instructions(code):
        if i.opname in _NOT_KEYED:
            continue
        jump = i.opcode in dis.hasjrel or i.opcode in dis.hasjabs or "JUMP" in i.opname or i.opname == "FOR_ITER"
        ops.append((i.opname, None if jump else i.argval.co_name if hasattr(i.argval, "co_code") else repr(i.argval)))
    return tuple(ops), code.co_varnames


def code_keys_of_file(path):
    """The code_key of every function of a Python file, compiled but not run (to recognise a copy of its code); an
    empty set for a file that cannot be read or compiled."""
    try:
        keys, todo = set(), [compile(open(path, encoding="utf-8").read(), str(path), "exec")]
    except (OSError, SyntaxError, ValueError, UnicodeDecodeError):
        return set()
    while todo:
        code = todo.pop()
        todo += [c for c in code.co_consts if hasattr(c, "co_code")]
        key = code_key(code)
        if code.co_name != "<module>" and len(key[0]) >= 15:
            keys.add(key)
    return keys


_ESCAPE = {"watch": None}      # what code_from(..., escapes=True) watches now (None: nothing)
_PROCESS_EVENTS = ("subprocess.Popen", "os.system", "os.posix_spawn", "os.fork", "os.forkpty", "os.exec", "os.spawn")


def _escape_hook(event, args, escape=_ESCAPE, processes=_PROCESS_EVENTS, os=os, stop=CodeFrom):
    """Audit hook, for the life of the kernel: inside code_from(..., escapes=True), opening a file of the watched places
    or starting another process stops the block (code run by another process escapes sys.monitoring). Everything it
    uses is bound when it is defined: it also runs while the kernel shuts down."""
    watch = escape["watch"]
    if watch is None:
        return
    if event == "open" and args and isinstance(args[0], (str, bytes, os.PathLike)):
        real = os.path.realpath(os.fsdecode(args[0]))
        if any(real == place or real.startswith(place + os.sep) for place in watch["places"]):
            watch["seen"].append("ouvre " + os.path.basename(real))
            raise stop
    elif event.startswith(processes):
        watch["seen"].append("lance un autre processus : " + event)
        raise stop


sys.addaudithook(_escape_hook)


@contextlib.contextmanager
def code_from(places, twins=(), escapes=False):
    """Inside the block, the start of any Python code (a function or a module body) of a file among places (files, or
    folders with everything under them), or of a function whose code_key is among twins (a copy of such a function),
    stops the block; yields the list of the names seen ("policy_gradient.importance_weights"). The learner's function
    must not run that code (mymath in a corrected function, the hidden module of a challenge). With escapes, opening a
    file of places or starting another process also stops it ("ouvre mb8_21.py", "lance un autre processus :
    subprocess.Popen"). Without sys.monitoring (Python < 3.12) or a free tool id, code is not watched."""
    places = [os.path.realpath(str(place)) for place in places]
    libraries = tuple({os.path.realpath(sys.prefix) + os.sep, os.path.realpath(sys.base_prefix) + os.sep})
    monitoring, tool, seen, inside = getattr(sys, "monitoring", None), None, [], {}
    for candidate in ((4, 3, 2) if monitoring is not None else ()):
        try:
            monitoring.use_tool_id(candidate, "code_from")
        except ValueError:                               # taken by another tool
            continue
        tool = candidate
        break
    if tool is not None:
        def on_start(code, offset):
            name = code.co_filename
            if name not in inside:
                real = os.path.realpath(name) if os.path.exists(name) else name
                if any(real == place or real.startswith(place + os.sep) for place in places):
                    inside[name] = True                  # watched
                elif twins and not real.startswith(libraries):
                    inside[name] = False                 # the learner's code: compared with twins
                else:
                    inside[name] = None                  # an installed library: not compared
            if inside[name]:
                seen.append(f"{os.path.splitext(os.path.basename(name))[0]}.{code.co_name}")
                raise CodeFrom
            if inside[name] is False and code_key(code) in twins:
                seen.append(f"une copie de {code.co_name}")
                raise CodeFrom
            return monitoring.DISABLE                    # other code: not watched again until the end of the block

        monitoring.register_callback(tool, monitoring.events.PY_START, on_start)
        monitoring.set_events(tool, monitoring.events.PY_START)
    if escapes:
        _ESCAPE["watch"] = {"places": places, "seen": seen}
    try:
        yield seen
    except CodeFrom:
        pass
    finally:
        _ESCAPE["watch"] = None
        if tool is not None:
            monitoring.set_events(tool, 0)
            monitoring.register_callback(tool, monitoring.events.PY_START, None)
            monitoring.free_tool_id(tool)
            monitoring.restart_events()


# The data of the paper exercises (02_exercices.md), read only
PHONE_1 = MappingProxyType({"rate": 0.1, "c": 18.0})                      # MB8.1: per minute; the voucher beyond c
POTS_2 = MappingProxyType({"theta": 0.75, "price": (0.0, 3.5, 5.0)})      # MB8.2: euros for 0, 1, 2 seedlings
TRUCK_3 = MappingProxyType({"stay": (0.86, 0.14), "leave": (0.0, 1.0), "sales_market": (3.0, 0.0),
                            "park": (0.3, 0.7), "sales_park": 4.0, "logits_market": (math.log(3), 0.0),
                            "gamma": 0.7})        # MB8.3: states (market, park); at the market, actions (stay, leave)
ROAD_4 = MappingProxyType({"P": frozen([[0.85, 0.15, 0.0], [0.0, 0.7, 0.3], [0.85, 0.0, 0.15]]), "gamma": 0.6,
                           "e": (3.0, 0.0, 2.0), "lambda": 0.7})          # MB8.4: (good, worn, works)
PANEL_5 = MappingProxyType({"q": (0.50, 0.25, 0.25), "p": (0.26, 0.41, 0.33), "f": (36.0, 50.0, 44.0),
                            "sample": (0, 0, 0, 1, 2, 2), "c": 1.6})       # MB8.5: (18-34, 35-59, 60 and more)
FOOTBALL_6 = MappingProxyType({"wins": frozen([[0, 2, 10], [5, 0, 1], [0, 3, 0]]), "proposed": (4.0, 3.0, 1.0),
                               "k": 32})                                  # MB8.6: clubs (A, B, C); row beats column
CANTEEN_7 = MappingProxyType({"pi_ref": (0.45, 0.40, 0.15), "r": (2.0, 4.0, 5.0), "betas": (1.0, 3.0)})   # MB8.7
LAUNDRY_11 = MappingProxyType({"theta": 2.5, "x": (3, 1, 4, 2, 5, 1, 3, 4)})   # MB8.11: customers per quarter hour

### Ex MB8.13 — L'estimateur score d'une dérivée sur un échantillon, avec une baseline constante ou la moyenne des autres tirages, et son erreur type · `score_function_gradient` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer l'estimateur score d'une dérivée et son erreur type, avec une baseline constante ou la moyenne des autres tirages, puis le comparer à la dérivée exacte de MB8.1.  
**Prérequis :** fiche §2 et §3 ; M02c (erreur type) ; Ex MB8.1 · **mymath :** `policy_gradient.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en MB6). Ouvre ta copie `mon_travail/mymath/policy_gradient.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.policy_gradient.<nom>` (par exemple `mymath.policy_gradient.importance_weights(lp, lq)`), ou donne-leur un nom court (`importance_weights = mymath.policy_gradient.importance_weights`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import math` et `import numpy as np`, et `from .markov import bellman_linear_solve` si tu reprends ta fonction de M26. NumPy et le module `math` sont permis ; de `np.linalg`, seulement `solve` et `matrix_rank` (dans `policy_gradient_exact`) ; ni SciPy (`scipy.optimize`, `scipy.special`…), ni scikit-learn : les tests comparent tes résultats à des sommes exactes, à des différences finies de valeurs calculées par `np.linalg.solve`, à `scipy.optimize.minimize` et à scikit-learn, et un test lit ton module. Rien n'y tire au hasard ; ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `score_function_gradient(f_values, scores, baseline=0.0)` (fiche §2 et §3), en suivant sa docstring : contrôle les arguments (au moins deux valeurs finies ; des scores à 1 ou 2 dimensions, avec une ligne par valeur ; une baseline qui est un nombre fini ou `"leave-one-out"`, pas un booléen) ; puis les termes $g_i = (f_i - b_i)\,s_i$, leur moyenne, et leur écart-type (`ddof=1`) divisé par $\sqrt n$. Lance le groupe de tests.

**L'usage.** Le notebook fournit `WAITS_13`, 2 000 attentes au téléphone de MB8.1 (taux $0{,}1$ par minute ; graine fixée), `F_13`, les $f(x_i) = \mathbf{1}\{x_i > 18\}$, et `S_13`, leurs scores. Avec ta fonction :
- a) l'estimation sans baseline, puis son erreur type (une liste de 2 nombres) ;
- b) l'estimation avec la moyenne des autres tirages, puis son erreur type (une liste de 2 nombres) ;
- c) le rapport de l'estimation avec la moyenne du groupe, `baseline=float(np.mean(F_13))`, à celle de b) (un nombre).

In [ ]:
# 2 000 waits on the phone of MB8.1 (rate 0.1 per minute; seed 8113), their values f(x_i) = 1{x_i > 18} and their
# scores in the rate, read only
WAITS_13 = frozen(np.random.default_rng(8113).exponential(1 / PHONE_1["rate"], 2000))
F_13 = frozen(WAITS_13 > PHONE_1["c"])
S_13 = frozen(1 / PHONE_1["rate"] - WAITS_13)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB8")  # reload your saved file

**Les tests** de `score_function_gradient` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB8, "test_score_function_gradient_ or test_module_uses_")

**L'usage** : 2 000 attentes au téléphone, sans baseline, avec la moyenne des autres tirages, avec la moyenne du groupe.

In [ ]:
answer_MB8_13a = answer_MB8_13b = answer_MB8_13c = ...
with wbm.attempt("MB8.13"):          # if you call score_function_gradient here: ⏳ while it is not written
    answer_MB8_13a = ...  # a) a list of 2 numbers, as computed: the estimate without baseline, then its standard error
    answer_MB8_13b = ...  # b) a list of 2 numbers, as computed: with the mean of the other draws, the estimate, then its standard error
    answer_MB8_13c = ...  # c) a number, as computed: the estimate with baseline=float(np.mean(F_13)), divided by the estimate of b)

In [ ]:
check_answer("MB8.13a", answer_MB8_13a, hint=library_hint("score_function_gradient"))
check_answer("MB8.13b", answer_MB8_13b, hint=library_hint("score_function_gradient"))
check_answer("MB8.13c", answer_MB8_13c, hint=library_hint("score_function_gradient"))

**Ce qu'il faut regarder** : la dérivée exacte (ta réponse à MB8.1 b) est-elle à moins de deux erreurs types de a) ? de b) ? Pourquoi l'erreur type de b) est-elle plus petite ? Pourquoi le rapport c) ne dépend-il pas du tirage ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB8.14 — Le gradient exact de la valeur d'une politique softmax tabulaire, par le système de Bellman · `policy_gradient_exact` 🔨 ★★ ⏱️ 35 min
**Objectif :** programmer le gradient exact de la valeur d'une politique softmax tabulaire, retrouver MB8.3, puis lire quels logits comptent pour un état de départ.  
**Prérequis :** fiche §4 ; M26 §4 et §5 (MDP, système de Bellman), M17 §7 (softmax) ; Ex MB8.3 · **mymath :** `policy_gradient.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB8.13 : écris la fonction dans ta copie `mon_travail/mymath/policy_gradient.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `policy_gradient_exact(P, r, logits, gamma)` (fiche §4), en suivant sa docstring : contrôle les arguments ; $\pi$, la softmax de chaque ligne des logits (retranche d'abord le maximum de la ligne) ; $\mathbf{P}_\pi$ et $\mathbf{r}_\pi$ ; $\mathbf{M}$ par `np.linalg.solve` ; $\mathbf{V}$, $Q$, puis la formule de la fiche §4 pour tous les $(s_0, s, a)$ d'un coup ; refuse $\gamma = 1$ quand $\mathbf{I} - \mathbf{P}_\pi$ est singulière. Lance le groupe de tests.

**L'usage.** Avec ta fonction :
- a) le camion de crêpes de MB8.3 (`TRUCK_3`, cellule de données), écrit comme un MDP à deux états (marché, parc) et deux actions (rester, partir) ; au parc, les deux actions font la même chose (mêmes lignes de $\mathbf{P}[a]$, même recette) : $V(\text{marché})$, $V(\text{parc})$, puis les dérivées de $V(\text{marché})$ par rapport aux deux logits du marché (rester, partir), puis celles de $V(\text{parc})$ (une liste de 6 nombres, à comparer à ta copie) ;
- b) le notebook fournit `P_14`, `R_14` et `LOGITS_14`, un MDP à trois états et deux actions, avec $\gamma = 0{,}6$ (`GAMMA_14`) : les dérivées de $V(0)$ par rapport aux six logits, état par état (les deux logits de l'état 0, puis de l'état 1, puis de l'état 2 ; une liste de 6 nombres) ;
- c) celles de $V(2)$, dans le même ordre (une liste de 6 nombres).

In [ ]:
# A Markov decision process of three states and two actions (the conventions of M26.14), read only
P_14 = frozen([[[0.7, 0.3, 0.0], [0.4, 0.6, 0.0], [0.5, 0.0, 0.5]], [[0.2, 0.8, 0.0], [0.9, 0.1, 0.0], [0.0, 0.3, 0.7]]])            # P_14[a, s, t]
R_14 = frozen([[1.0, 0.0, 2.0], [0.5, 2.0, -1.0]])            # R_14[a, s]
LOGITS_14 = frozen([[0.2, -0.3], [0.0, 0.6], [-0.5, 0.4]])  # LOGITS_14[s, a]
GAMMA_14 = 0.6

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB8")  # reload your saved file

**Les tests** de `policy_gradient_exact` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB8, "test_policy_gradient_exact_")

**L'usage** : le camion de crêpes de MB8.3, puis un MDP à trois états.

In [ ]:
answer_MB8_14a = answer_MB8_14b = answer_MB8_14c = ...
with wbm.attempt("MB8.14"):          # if you call policy_gradient_exact here: ⏳ while it is not written
    answer_MB8_14a = ...  # a) a list of 6 numbers, as computed: V(market), V(park), the 2 derivatives of V(market), the 2 of V(park)
    answer_MB8_14b = ...  # b) a list of 6 numbers, as computed: the derivatives of V(0), state by state
    answer_MB8_14c = ...  # c) a list of 6 numbers, as computed: the derivatives of V(2), state by state

In [ ]:
check_answer("MB8.14a", answer_MB8_14a, hint=library_hint("policy_gradient_exact"))
check_answer("MB8.14b", answer_MB8_14b, hint=library_hint("policy_gradient_exact"))
check_answer("MB8.14c", answer_MB8_14c, hint=library_hint("policy_gradient_exact"))

**Ce qu'il faut regarder** : pourquoi, en b), les dérivées par rapport aux logits de l'état 2 sont-elles nulles, et pas en c) ? Pourquoi les deux dérivées d'un même état ont-elles une somme nulle ? Pourquoi, en a), les logits du parc ne comptent-ils pas ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB8.15 — Des poids d'importance calculés en log, auto-normalisés ou tronqués, et la taille effective de l'échantillon · `importance_weights` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer des poids d'importance calculés en logarithmes, tronqués ou auto-normalisés, avec leur taille effective, puis les essayer sur des densités qui sortent des flottants.  
**Prérequis :** fiche §6 ; M05a (log-sum-exp) ; Ex MB8.5 · **mymath :** `policy_gradient.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB8.13 : écris la fonction dans ta copie `mon_travail/mymath/policy_gradient.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `importance_weights(log_target, log_proposal, self_normalize=False, max_weight=None)` (fiche §6), en suivant sa docstring : contrôle les arguments ; les différences $d_i = \ln p(x_i) - \ln q(x_i)$, ramenées à $\ln c$ au plus avec `max_weight` $= c$ ; la taille effective à partir de $e^{d_i - \max d}$ ; puis les poids $e^{d_i}$, ou divisés par leur somme. Lance le groupe de tests.

**L'usage.** Avec ta fonction :
- a) les six panélistes de MB8.5 (`PANEL_5`, cellule de données ; les $\ln p$ et $\ln q$ de leurs classes) : le plus grand poids auto-normalisé, puis la taille effective (une liste de 2 nombres, à comparer à ta copie) ;
- b) le notebook fournit `LOG_P_15` et `LOG_Q_15`, les log-densités de cinq tirages d'une proposition en dimension 900, de l'ordre de $-1\,300$ : calcule d'abord `np.exp(LOG_P_15) / np.exp(LOG_Q_15)` et regarde ce que rend NumPy ; puis, avec `self_normalize=True`, le plus grand poids normalisé et la taille effective (une liste de 2 nombres).

In [ ]:
# Five draws of a proposal in dimension 900: the logarithms of the target and proposal densities there, read only
LOG_P_15 = frozen([-1311.7703, -1280.1143, -1290.4471, -1286.6555, -1244.938])
LOG_Q_15 = frozen([-1310.5853, -1278.9503, -1289.6596, -1286.9464, -1244.907])

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB8")  # reload your saved file

**Les tests** de `importance_weights` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB8, "test_importance_weights_")

**L'usage** : les six panélistes de MB8.5, puis cinq tirages en dimension 900.

In [ ]:
answer_MB8_15a = answer_MB8_15b = naive_15 = ...
with wbm.attempt("MB8.15"):          # if you call importance_weights here: ⏳ while it is not written
    answer_MB8_15a = ...  # a) a list of 2 numbers, as computed: the six panelists, the largest self-normalized weight, then the effective size
    naive_15 = ...        # b) first: np.exp(LOG_P_15) / np.exp(LOG_Q_15), and look at what NumPy returns
    answer_MB8_15b = ...  # b) a list of 2 numbers, as computed: with self_normalize=True, the largest weight, then the effective size

In [ ]:
check_answer("MB8.15a", answer_MB8_15a, hint=library_hint("importance_weights"))
check_answer("MB8.15b", answer_MB8_15b, hint=library_hint("importance_weights"))

**Ce qu'il faut regarder** : pourquoi le quotient direct ne donne-t-il rien d'utilisable ? Pourquoi les poids de b) sont-ils pourtant bien définis ? Que dit la taille effective trouvée, pour cinq tirages ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB8.16 — Les forces d'un modèle de Bradley-Terry par maximum de vraisemblance (l'algorithme de Zermelo), centrées, et le refus d'un tournoi où elles n'existent pas · `bradley_terry_fit` 🔨 ★★ ⏱️ 35 min
**Objectif :** programmer les forces d'un modèle de Bradley-Terry par l'algorithme de Zermelo, retrouver MB8.6, puis ajuster une saison de six clubs et voir quand les forces n'existent pas.  
**Prérequis :** fiche §7 ; M16 (chemins d'un graphe), M13a (sigmoïde) ; Ex MB8.6 · **mymath :** `policy_gradient.py` · **Parcours :** C, E

> Même mode d'emploi qu'en MB8.13 : écris la fonction dans ta copie `mon_travail/mymath/policy_gradient.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `bradley_terry_fit(wins, tol=1e-10, max_iter=10_000)` (fiche §7), en suivant sa docstring : contrôle les arguments ; la forte connexité (la matrice booléenne « $i$ atteint $j$ » et ses produits par elle-même, M16) ; puis les balayages de Zermelo sur $p = e^\beta$, centrés à chaque balayage, jusqu'à ce que le plus grand changement de $\beta$ passe sous `tol`. Lance le groupe de tests.

**L'usage.** Avec ta fonction :
- a) le tournoi de MB8.6 (`FOOTBALL_6`, cellule de données) : les forces centrées de A, B, C (une liste de 3 nombres, à comparer à ta copie) ;
- b) le notebook fournit `WINS_16`, une saison de six clubs, qui n'ont pas tous joué le même nombre de matchs : les six forces, puis la probabilité que le club 0 batte le club 5 (une liste de 7 nombres) ;
- c) dans une copie de `WINS_16`, mets à 0 toutes les défaites du club 1 (sa colonne) : que fait ta fonction ? (la vérification attend une `ValueError`).

In [ ]:
# A season of six table-football clubs (0 to 5): WINS_16[i, j] is the number of wins of club i against club j, read only
WINS_16 = frozen([[0, 3, 4, 2, 5, 6], [2, 0, 3, 3, 4, 5], [1, 2, 0, 3, 4, 4], [3, 2, 2, 0, 3, 3], [0, 1, 1, 2, 0, 2],
 [1, 0, 1, 2, 3, 0]])

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB8")  # reload your saved file

**Les tests** de `bradley_terry_fit` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB8, "test_bradley_terry_fit_")

**L'usage** : le tournoi de MB8.6, une saison de six clubs, puis un club sans défaite.

In [ ]:
answer_MB8_16a = answer_MB8_16b = wins_16c = ...
with wbm.attempt("MB8.16"):          # if you call bradley_terry_fit here: ⏳ while it is not written
    answer_MB8_16a = ...  # a) a list of 3 numbers, as computed: the centred strengths of A, B, C
    answer_MB8_16b = ...  # b) a list of 7 numbers, as computed: the six strengths, then the probability that club 0 beats club 5
wins_16c = ...            # c) a copy of WINS_16 whose column 1 (the defeats of club 1) is set to 0

In [ ]:
check_answer("MB8.16a", answer_MB8_16a, hint=library_hint("bradley_terry_fit"))
check_answer("MB8.16b", answer_MB8_16b, hint=library_hint("bradley_terry_fit"))

with checked("MB8.16"):                                     # c) a club that never lost
    if not filled(wins_16c):
        print("⏳ Ex MB8.16c : pas encore fait (remplace `...` par ta copie de WINS_16).")
    else:
        mine_16c, expected_16c = as_floats(wins_16c), np.array(WINS_16)
        expected_16c[:, 1] = 0.0
        if mine_16c is None or mine_16c.shape != expected_16c.shape or not np.array_equal(mine_16c, expected_16c):
            print("❌ Ex MB8.16c : wins_16c doit être une copie de WINS_16 (6 lignes, 6 colonnes) dont seule la colonne 1, "
                  "les défaites du club 1, vaut 0")
        else:
            refuses("MB8.16c", "bradley_terry_fit(wins_16c), le club 1 sans défaite",
                    lambda: mymath.policy_gradient.bradley_terry_fit(wins_16c))

**Ce qu'il faut regarder** : la probabilité que le club 0 batte le club 5 vaut-elle la part de ses victoires contre lui, 6 sur 7 ? Pourquoi le modèle ne recopie-t-il pas cette part ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie B · Prédire, mesurer

Fiche §3 et §6. Trois baselines pour une même dérivée ; un événement rare, l'échantillonnage d'importance et des poids tronqués.

### Ex MB8.17 — Prédire l'ordre des écarts-types de l'estimateur score d'une même dérivée, sans baseline, avec la moyenne des autres tirages et avec la baseline optimale, puis les mesurer 🔮 ★ ⏱️ 20 min
**Objectif :** prédire l'ordre des dispersions de l'estimateur score d'une même dérivée pour trois baselines, puis les mesurer.  
**Prérequis :** fiche §3 ; Ex MB8.2, MB8.13 · **Parcours :** C, E

$X$ suit une loi de Poisson de moyenne $\theta = 3{,}5$ ; on estime $\frac{\mathrm{d}}{\mathrm{d}\theta}P_\theta(X \ge 5)$ par l'estimateur score sur $n = 40$ tirages, de trois façons : sans baseline, avec la moyenne des autres tirages, et avec la baseline optimale $b^\star$, que le notebook calcule par des sommes exactes sur la loi (`B_STAR_17`). On répète l'estimation 2 000 fois : le notebook fournit `DRAWS_17`, 2 000 échantillons de 40 tirages, un par ligne (le score d'un tirage est $\frac{x}{\theta} - 1$). **Avant d'exécuter quoi que ce soit**, prédis (une lettre chacune) :
- a) l'ordre des trois écarts-types : (A) ils sont égaux, puisque les trois estimations ont la même espérance ; (B) sans baseline < moyenne des autres < baseline optimale ; (C) sans baseline > moyenne des autres > baseline optimale ;
- b) le rapport de l'écart-type sans baseline à celui avec $b^\star$ : (A) entre 1 et $1{,}5$ ; (B) entre $1{,}5$ et $2{,}5$ ; (C) plus de $2{,}5$.

Puis, une fois tes prédictions écrites et vérifiées, mesure avec `score_function_gradient` :
- c) les trois écarts-types des 2 000 estimations, dans l'ordre de a) (une liste de 3 nombres).

La vérification de l'expérience ne s'exécute qu'une fois tes deux prédictions écrites.

In [ ]:
# X ~ Poisson of mean 3.5 and f(x) = 1{x >= 5}: 2 000 samples of n = 40 draws (seed 8117), one per row, and the optimal
# baseline b* = E[f s²] / E[s²] by exact sums over the law, read only
THETA_17, N_17 = 3.5, 40
_rng_17 = np.random.default_rng(8117)
DRAWS_17 = frozen(np.array([_rng_17.poisson(THETA_17, N_17) for _ in range(2000)]))
del _rng_17
_support_17 = np.arange(80)
_law_17 = stats.poisson.pmf(_support_17, THETA_17)
_f_17, _s_17 = (_support_17 >= 5).astype(float), _support_17 / THETA_17 - 1
B_STAR_17 = float(np.sum(_law_17 * _f_17 * _s_17 ** 2) / np.sum(_law_17 * _s_17 ** 2))
del _support_17, _law_17, _f_17, _s_17

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_MB8_17a = ...  # a) "A", "B" or "C": (A) equal, (B) none < others < optimal, (C) none > others > optimal
prediction_MB8_17b = ...  # b) "A", "B" or "C": the standard deviation none / optimal: (A) 1 to 1.5, (B) 1.5 to 2.5, (C) more than 2.5

In [ ]:
for letter, prediction in zip("ab", [prediction_MB8_17a, prediction_MB8_17b]):
    if filled(prediction) and not isinstance(prediction, str):      # a number, a list...: not yet a prediction
        print(f'❌ Ex MB8.17{letter} : une prédiction est une lettre, "A", "B" ou "C", reçu {shown(prediction)}')
    else:
        wbm.check(f"MB8.17{letter}", prediction)

**L'expérience.** Écris-la une fois tes deux prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
answer_MB8_17c = ...
with wbm.attempt("MB8.17"):          # your score_function_gradient: ⏳ while it is not written
    answer_MB8_17c = ...  # c) a list of 3 numbers, as computed: the standard deviations of the 2 000 estimates, without baseline, with the mean of the other draws, with B_STAR_17

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_MB8_17a, prediction_MB8_17b]):
    print("⏳ Écris d'abord tes deux prédictions, puis exécute à nouveau.")
else:
    check_answer("MB8.17c", answer_MB8_17c, hint=library_hint("score_function_gradient"))

**Ce qu'il faut regarder** (après l'expérience) : les trois moyennes des estimations sont-elles proches de la même valeur ? $b^\star$ est-il proche de $P_\theta(X \ge 5)$ ? Pourquoi la moyenne des autres tirages, proche de $\mathbb{E}[f]$, ne fait-elle pas aussi bien que $b^\star$ ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB8.18 — Un événement rare, P(Z > 4) pour Z normale centrée réduite : Monte-Carlo naïf contre l'échantillonnage d'importance sous N(4, 1), la taille effective, puis des poids tronqués à c : le biais et la variance selon c 🔬 ★★ ⏱️ 30 min
**Objectif :** estimer la probabilité d'un événement rare par Monte-Carlo naïf et par échantillonnage d'importance, lire la taille effective, puis mesurer ce que la troncature des poids fait au biais et à la dispersion.  
**Prérequis :** fiche §6 ; M20 §5 (Horvitz-Thompson) ; Ex MB8.5, MB8.15 · **Parcours :** C

$Z$ suit la loi normale centrée réduite, et l'on estime $P(Z > 4)$, connue par ailleurs (`scipy.stats.norm.sf(4)`, que le notebook range dans `TAIL_18`).
- a) Monte-Carlo naïf sur les 100 000 tirages `DRAWS_18A` : l'estimation, puis le nombre de tirages au-delà de 4 (une liste de 2 nombres) ;
- b) échantillonnage d'importance sous la proposition $q = \mathcal{N}(4, 1)$, avec les 10 000 tirages `DRAWS_18B` et `importance_weights` (les log-densités par `scipy.stats.norm.logpdf`) : l'estimation divisée par la vraie valeur, l'erreur type relative (l'erreur type divisée par la vraie valeur), puis la taille effective des 10 000 poids (une liste de 3 nombres) ;
- c) des poids tronqués à $c$, pour les quatre niveaux $c = 10^{-4}$, $2 \cdot 10^{-4}$, $3{,}35 \cdot 10^{-4}$ et 1 (`C_18`) : sur les 400 répétitions de 10 000 tirages `DRAWS_18C` (une par ligne), le biais relatif (la moyenne des 400 estimations moins la vraie valeur, divisée par la vraie valeur) et l'écart-type relatif (l'écart-type des 400 estimations divisé par la vraie valeur) pour chaque $c$, en deux tableaux `rel_bias_18` et `rel_sd_18` ; écris `draw_18(ax, c_values, rel_bias, rel_sd)`, qui les trace en deux courbes en fonction de $c$ dans l'axe `ax` (avec une légende et des noms d'axes ; une échelle logarithmique en $c$ se lit mieux), sans ouvrir de figure ; relève les biais relatifs pour $c = 10^{-4}$ et $2 \cdot 10^{-4}$ (une liste de 2 nombres).

La vérification appelle `draw_18` avec tes deux tableaux et affiche la figure.

In [ ]:
# Z ~ N(0, 1), the event Z > 4 and its probability; the draws of the experiment, read only: a) 100 000 draws of N(0, 1)
# (seed 8118); b) 10 000 draws of the proposal q = N(4, 1) (seed 8119); c) 400 rows of 10 000 draws of q (seed 8120)
TAIL_18 = float(stats.norm.sf(4.0))
DRAWS_18A = frozen(np.random.default_rng(8118).normal(size=100_000))
DRAWS_18B = frozen(np.random.default_rng(8119).normal(4.0, 1.0, 10_000))
_rng_18 = np.random.default_rng(8120)
DRAWS_18C = frozen(np.array([_rng_18.normal(4.0, 1.0, 10_000) for _ in range(400)]))
del _rng_18
C_18 = (1e-4, 2e-4, 3.35e-4, 1.0)            # the truncation levels of c)

In [ ]:
def draw_18(ax, c_values, rel_bias, rel_sd):
    """In ax: the relative bias and the relative standard deviation of the estimate against the truncation level c (two
    curves, with a legend and axis labels). Opens no figure."""
    raise NotImplementedError("draw_18 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB8_18a = answer_MB8_18b = answer_MB8_18c = rel_bias_18 = rel_sd_18 = ...
with wbm.attempt("MB8.18"):          # if you call importance_weights here: ⏳ while it is not written
    answer_MB8_18a = ...  # a) a list of 2 numbers: on DRAWS_18A, the naive estimate, then the number of draws beyond 4
    answer_MB8_18b = ...  # b) a list of 3 numbers, as computed: on DRAWS_18B, the estimate / TAIL_18, the relative standard error, the effective size
    rel_bias_18 = ...     # c) the 4 relative biases, one per level of C_18 (the 400 rows of DRAWS_18C)
    rel_sd_18 = ...       # c) the 4 relative standard deviations, one per level of C_18
    answer_MB8_18c = ...  # c) a list of 2 numbers, as computed: the relative biases for c = 1e-4, then 2e-4

In [ ]:
HIDDEN_C18 = {'id_sizes': 'MB8.18d'}   # the sub-ID of the orders of size of the deviations

check_answer("MB8.18a", answer_MB8_18a)
check_answer("MB8.18b", answer_MB8_18b, hint=library_hint("importance_weights"))
with checked("MB8.18"):
    results_18 = []
    report_18 = reporter(results_18, stop=False)
    curves_18 = [as_floats(rel_bias_18), as_floats(rel_sd_18)] if filled(rel_bias_18, rel_sd_18) else None
    if curves_18 is None:
        print("⏳ Ex MB8.18 : rel_bias_18 et rel_sd_18 ne sont pas encore calculés.")
        raise StopCheck
    if any(c is None or c.shape != (len(C_18),) or not np.all(np.isfinite(c)) for c in curves_18):
        print(f"❌ Ex MB8.18 : rel_bias_18 et rel_sd_18 sont deux tableaux de {len(C_18)} nombres finis, un par niveau c "
              f"de C_18, reçu {shown(rel_bias_18)} et {shown(rel_sd_18)}")
        raise StopCheck
    fig_18, (ax_18, spare_18) = plt.subplots(1, 2, figsize=(11, 4))            # spare_18: the current axes
    try:
        opened_18 = drawn(lambda: call_safely("Ex MB8.18 : draw_18", draw_18, ax_18, C_18, curves_18[0], curves_18[1]))
        found_18 = []
        for values_18 in curves_18:
            found_18.append(next((line for line in ax_18.lines
                                  if np.asarray(line.get_xdata(), dtype=float).shape == (len(C_18),)
                                  and np.allclose(np.asarray(line.get_xdata(), dtype=float), C_18, rtol=1e-12, atol=0)
                                  and np.allclose(np.asarray(line.get_ydata(), dtype=float), values_18, rtol=1e-12,
                                                  atol=1e-15)), None))
        report_18(all(line is not None for line in found_18),
                  "deux courbes en fonction de c : le biais relatif et l'écart-type relatif",
                  "il faut deux courbes dans ax, en fonction de c (les niveaux de C_18, dans l'ordre) : le biais relatif "
                  "(rel_bias_18) et l'écart-type relatif (rel_sd_18)")
        labels_18 = [line.get_label().lower() if line is not None else "" for line in found_18]
        report_18(all(line is not None for line in found_18)
                  and not (any(word in labels_18[1] for word in ("biais", "bias"))
                           and not any(word in labels_18[0] for word in ("biais", "bias"))),
                  "chaque courbe porte son nom",
                  "les noms de la légende sont échangés : la courbe de rel_bias_18 porte "
                  f"« {labels_18[0]} », celle de rel_sd_18 « {labels_18[1]} »"
                  if all(line is not None for line in found_18) else
                  "les noms des courbes se jugent sur les deux courbes : trace-les d'abord dans ax")
        legend_18 = ax_18.get_legend()
        report_18(legend_18 is not None and all(line is not None and line.get_label() and not
                                                line.get_label().startswith("_") for line in found_18)
                  and bool(ax_18.get_xlabel()) and bool(ax_18.get_ylabel()),
                  "une légende qui nomme chaque courbe, et des noms d'axes",
                  "il faut une légende (label=…, puis ax.legend()) qui nomme chaque courbe, et des noms d'axes"
                  if all(line is not None for line in found_18) else
                  "la légende et les noms d'axes se jugent sur les deux courbes : trace-les d'abord dans ax")
        elsewhere_18 = bool(spare_18.lines or spare_18.collections or spare_18.get_legend())
        report_18(not opened_18 and not elsewhere_18, "draw_18 dessine dans l'axe reçu",
                  "draw_18 doit dessiner dans l'axe reçu (ax.plot…), sans ouvrir de figure ni dessiner dans l'axe "
                  "courant (plt.plot, plt.legend…)")
        fig_18.delaxes(spare_18)
    except BaseException:
        plt.close(fig_18)
        raise
    fig_18.tight_layout()
    plt.show()
    report_18(passes("MB8.18c", curves_18[0][:2].tolist()) and bool(np.all(np.abs(curves_18[0][2:]) < 0.01)),
              "les biais relatifs sont ceux de l'estimation",
              "les biais relatifs ne sont pas ceux de l'estimation : pour chaque c, la moyenne des 400 estimations moins "
              "TAIL_18, divisée par TAIL_18")
    sizes_18 = np.round(np.log10(np.abs(curves_18[1]) + 1e-300)).tolist()
    report_18(bool(np.all(curves_18[1] > 0)) and passes(HIDDEN_C18["id_sizes"], sizes_18),
              "les écarts-types relatifs ont l'ordre de grandeur attendu",
              "les écarts-types relatifs n'ont pas l'ordre de grandeur de ceux de l'estimation : l'écart-type des 400 "
              "estimations (une par ligne de DRAWS_18C), divisé par TAIL_18")
    verdict("MB8.18", all(results_18) and len(results_18) == 6, "ta figure et tes mesures de c) sont justes.",
            "relis les lignes ❌ ci-dessus.")
if filled(answer_MB8_18c) and filled(rel_bias_18) and as_floats(answer_MB8_18c) is not None \
        and as_floats(answer_MB8_18c).shape == (2,) and as_floats(rel_bias_18) is not None \
        and as_floats(rel_bias_18).shape == (len(C_18),) \
        and not np.allclose(as_floats(answer_MB8_18c), as_floats(rel_bias_18)[:2], rtol=1e-9, atol=1e-12):
    print("❌ Ex MB8.18c : c) relève les deux premiers biais relatifs de rel_bias_18 (c = 10⁻⁴, puis 2·10⁻⁴), sans les "
          "arrondir")
else:
    check_answer("MB8.18c", answer_MB8_18c, hint=library_hint("importance_weights"))

**Ce qu'il faut regarder** : pourquoi le Monte-Carlo naïf ne donne-t-il presque rien ? Que vaut le poids $\frac pq$ d'un tirage $z$, et quel est le plus grand poids d'un tirage au-delà de 4 ? Pourquoi la taille effective de b) est-elle si petite, alors que l'estimation est précise ? À partir de quel $c$ la troncature ne biaise-t-elle plus ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie C · Déboguer, tester

Fiche §1, §3, §4, §6 et §7. Quatre erreurs, qui passent une vérification trop faible ; deux tests de propriété du gradient exact. Le code du collègue s'appuie sur la référence (`ref_pg`), pour ne pas dépendre de ton code.

### Ex MB8.19 — Chasse à l'erreur : le score d'une loi exponentielle écrit pour la moyenne au lieu du taux, des poids d'importance inversés (q/p), une baseline qui contient le tirage lui-même, des forces de deux tournois sans joueur commun comparées sur une même échelle 🐛 ★ ⏱️ 25 min
**Objectif :** reconnaître quatre erreurs à leurs symptômes, qui passent une vérification trop faible, puis les corriger.  
**Prérequis :** fiche §1, §3, §6 et §7 ; Ex MB8.13, MB8.15, MB8.16 · **Parcours :** C, E

Un collègue a écrit quatre fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `score_exp_bad(x, theta)` doit rendre le score $s_\theta(x)$ de la loi exponentielle de taux $\theta$ en chaque point de `x`. Sur 100 000 tirages au taux $0{,}4$ (`X_19A`), la moyenne des scores est loin de 0. Il l'a vérifiée au taux 1, où la moyenne des scores est proche de 0.
2. `weights_bad(log_p, log_q)` doit rendre les poids d'importance de tirages de $q$. Avec la cible $\mathcal{N}(0{,}8 ;\ 1)$ et la proposition $\mathcal{N}(0 ;\ 1{,}5^2)$ (`X_19B`, `LOG_P_19B`, `LOG_Q_19B`), la moyenne des $w_ix_i$, qui devrait estimer $\mathbb{E}_p[X] = 0{,}8$, est énorme et négative. Il l'a vérifiée avec la même loi pour $p$ et pour $q$.
3. `estimate_bad(f, s)` doit rendre l'estimateur score d'un batch, avec pour chaque tirage une baseline calculée sur le batch, et rester sans biais. Sur 20 000 batches de 4 tirages (`BATCHES_19C` : loi de Poisson de moyenne $1{,}5$, $f(x) = \mathbf{1}\{x \ge 2\}$, le score $\frac{x}{1{,}5} - 1$), la moyenne des estimations reste nettement sous la dérivée exacte, $1{,}5\,e^{-1{,}5}$. Il l'a vérifiée sur un seul batch de 10 000 tirages.
4. `strongest_bad(wins_a, wins_b)` doit dire quel est le club le plus fort de deux tournois réunis, A (clubs A0, A1, A2 ; `TOURNAMENT_A`) et B (clubs B0, B1, B2 ; `TOURNAMENT_B`), ou lever une `ValueError` quand les données ne permettent pas de le dire. Quand on ajoute au tournoi B un club faible, B3, sans rien changer aux matchs des autres (`TOURNAMENT_B3`), sa réponse change. Il l'a vérifiée en lui donnant deux fois le même tableau de matchs : elle a bien désigné le meilleur club de ce tableau.

À faire : trouve et corrige chaque erreur dans le code du collègue lui-même : les fonctions corrigées s'appellent `score_exp_fixed`, `weights_fixed`, `estimate_fixed` et `strongest_fixed`, avec les mêmes arguments, et n'appellent ni `mymath` ni la référence `ref_pg` dont se sert le collègue (la vérification le contrôle, copies de leur code comprises). Puis réponds :
- a) la moyenne des scores de `X_19A` au taux $0{,}4$, avec la fonction corrigée, puis avec la fonction fautive (une liste de 2 nombres) ;
- b) l'estimation de $\mathbb{E}_p[X]$, la moyenne des $w_ix_i$, corrigée puis fautive (une liste de 2 nombres) ;
- c) la moyenne des 20 000 estimations, corrigée puis fautive (une liste de 2 nombres) ;
- d) les réponses de la fonction fautive avant, puis après l'ajout de B3 (une liste de 2 noms de clubs, comme `"A1"`).

Pour chacune des quatre erreurs, dis aussi, dans ta copie, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie aussi tes fonctions corrigées sur des cas cachés : d'autres taux, d'autres log-densités, d'autres batches, d'autres paires de tournois, et des tableaux qu'elles ne doivent pas modifier.

In [ ]:
# The colleague's code
def score_exp_bad(x, theta):
    """The score s_θ(x) = ∂_θ ln p_θ(x) of the exponential law of rate theta, at each point of x."""
    x = np.asarray(x, dtype=float)
    return (x - theta) / theta ** 2


def weights_bad(log_p, log_q):
    """The importance weights p(x_i) / q(x_i) of draws x_i of q, from the logarithms of the two densities."""
    return np.exp(np.asarray(log_q, dtype=float) - np.asarray(log_p, dtype=float))


def estimate_bad(f, s):
    """The score estimate (1/n) Σ (f_i - b_i) s_i of a batch, each baseline b_i computed on the batch, unbiased."""
    f, s = np.asarray(f, dtype=float), np.asarray(s, dtype=float)
    return float(np.mean((f - f.mean()) * s))


def strongest_bad(wins_a, wins_b):
    """The strongest club of two tournaments put together, A (clubs A0, A1, ...) and B (clubs B0, B1, ...), as "A0",
    "B2"...; ValueError when the data do not allow to say."""
    beta_a = ref_pg.bradley_terry_fit(wins_a)            # the strengths of each tournament (the reference's fit)
    beta_b = ref_pg.bradley_terry_fit(wins_b)
    if beta_a.max() >= beta_b.max():
        return f"A{int(np.argmax(beta_a))}"
    return f"B{int(np.argmax(beta_b))}"


# The data, read only: (1) 100 000 draws at the rate 0.4 (seed 8121); (2) 50 000 draws of q = N(0, 1.5²) (seed 8122)
# and the logarithms of the target p = N(0.8, 1) and of q there; (3) 20 000 batches of 4 draws of a Poisson law of mean
# 1.5 (seed 8123); (4) the tournaments A and B (wins of the row against the column), and B with a weak club B3
X_19A = frozen(np.random.default_rng(8121).exponential(1 / 0.4, 100_000))
X_19B = frozen(np.random.default_rng(8122).normal(0.0, 1.5, 50_000))
LOG_P_19B, LOG_Q_19B = frozen(stats.norm.logpdf(X_19B, 0.8, 1.0)), frozen(stats.norm.logpdf(X_19B, 0.0, 1.5))
BATCHES_19C = frozen(np.random.default_rng(8123).poisson(1.5, (20_000, 4)))
TOURNAMENT_A = frozen([[0, 5, 6], [3, 0, 4], [2, 4, 0]])
TOURNAMENT_B = frozen([[0, 4, 3], [4, 0, 5], [5, 3, 0]])
TOURNAMENT_B3 = frozen([[0, 4, 3, 5], [4, 0, 5, 5], [5, 3, 0, 5], [1, 1, 1, 0]])

# The colleague's checks: all pass
_check_x_19 = np.random.default_rng(8121).exponential(1.0, 100_000)
print("(1)", bool(abs(np.mean(score_exp_bad(_check_x_19, 1.0))) < 0.025), "(attendu True : au taux 1, la moyenne des "
      "scores est proche de 0)")
print("(2)", bool(np.allclose(weights_bad(LOG_Q_19B, LOG_Q_19B), 1.0)), "(attendu True : avec la même loi pour p et "
      "pour q, tous les poids valent 1)")
_check_batch_19 = np.random.default_rng(8126).poisson(1.5, 10_000)
print("(3)", bool(abs(estimate_bad(_check_batch_19 >= 2, _check_batch_19 / 1.5 - 1) - 1.5 * math.exp(-1.5)) < 0.025),
      "(attendu True : sur un batch de 10 000 tirages, l'estimation est proche de la dérivée exacte)")
print("(4)", strongest_bad(TOURNAMENT_A, TOURNAMENT_A) == "A0", "(attendu True : deux fois le même tableau de matchs, "
      "le meilleur club de ce tableau)")
del _check_x_19, _check_batch_19

In [ ]:
def score_exp_fixed(x, theta):
    """The score s_θ(x) = ∂_θ ln p_θ(x) of the exponential law of rate theta, at each point of x."""
    raise NotImplementedError("score_exp_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def weights_fixed(log_p, log_q):
    """The importance weights p(x_i) / q(x_i) of draws x_i of q, from the logarithms of the two densities."""
    raise NotImplementedError("weights_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def estimate_fixed(f, s):
    """The score estimate (1/n) Σ (f_i - b_i) s_i of a batch, each baseline b_i computed on the batch, unbiased."""
    raise NotImplementedError("estimate_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def strongest_fixed(wins_a, wins_b):
    """The strongest club of two tournaments put together, A (clubs A0, A1, ...) and B (clubs B0, B1, ...), as "A0",
    "B2"...; ValueError when the data do not allow to say."""
    raise NotImplementedError("strongest_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB8_19a = answer_MB8_19b = answer_MB8_19c = answer_MB8_19d = ...
with wbm.attempt("MB8.19"):          # your four functions: ⏳ while they are not written
    answer_MB8_19a = ...  # a) a list of 2 numbers, as computed: the mean of the scores of X_19A at the rate 0.4 by score_exp_fixed, then by the WRONG score_exp_bad
    answer_MB8_19b = ...  # b) a list of 2 numbers, as computed: the mean of w_i x_i on X_19B by weights_fixed, then by the WRONG weights_bad
    answer_MB8_19c = ...  # c) a list of 2 numbers, as computed: the mean of the 20 000 estimates on BATCHES_19C by estimate_fixed, then by the WRONG estimate_bad
    answer_MB8_19d = ...  # d) a list of 2 names: the answers of the WRONG strongest_bad, before then after adding B3

In [ ]:
for letter, answer in zip("abc", [answer_MB8_19a, answer_MB8_19b, answer_MB8_19c]):
    check_answer(f"MB8.19{letter}", answer)
if not filled(answer_MB8_19d):
    print("⏳ Ex MB8.19d : pas encore fait.")
elif not (isinstance(answer_MB8_19d, (list, tuple)) and len(answer_MB8_19d) == 2
          and all(isinstance(name, str) for name in answer_MB8_19d)):
    print(f"❌ Ex MB8.19d : une liste de 2 noms de clubs (des chaînes comme \"A1\"), reçu {shown(answer_MB8_19d)}")
else:
    wbm.check("MB8.19d", " ".join(name.strip() for name in answer_MB8_19d))

CASES_C19 = {'scores': [(0.53, [0.87, 2.41, 5.06, 0.17, 3.38]), (3.65, [0.091, 0.462, 0.257, 0.738, 0.118]),
            (8.6, [0.0213, 0.164, 0.0587, 0.311, 0.0942])],
 'weights': [([-1.37, -0.62, -2.91, -0.23], [-0.97, -1.04, -1.33, -0.47]),
             ([-3.61, -2.07, -4.83], [-2.94, -2.88, -3.17])],
 'batches': [([0.7, 1.85], [0.43, -1.13]), ([1.0, 0.0, 1.0], [0.62, -0.87, 1.31]),
             ([2.35, 0.4, 1.65, 3.15, 0.9], [0.27, -0.79, 0.55, 1.42, -0.63]),
             ([1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0], [0.38, 1.27, -0.89, 0.71, -1.33, -0.52, 2.06])],
 'pairs': [([[0, 3], [2, 0]], [[0, 1, 4], [2, 0, 3], [3, 1, 0]]),
           ([[0, 2, 1, 3], [1, 0, 2, 2], [3, 1, 0, 1], [2, 2, 3, 0]], [[0, 6], [5, 0]])],
 'id_scores': 'MB8.19e',
 'id_weights': 'MB8.19f',
 'id_batches': 'MB8.19g'}   # the hidden cases, and the sub-IDs of the values expected

results_19, parts_19 = [], []
MYMATH_C19 = list(getattr(mymath, "__path__", [])) + list(getattr(REF, "__path__", []))   # the folders of mymath
TWINS_C19 = set()                                  # the functions of policy_gradient.py (yours, the reference's)
for _folder_c19 in MYMATH_C19:
    TWINS_C19 |= code_keys_of_file(os.path.join(_folder_c19, "policy_gradient.py"))


def report_19(ok, success, failure):
    results_19.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_19(part, name, function, *args):
    """function(*args), without mymath; an error of the learner's code becomes one ❌ line that names the function."""
    try:
        with code_from(MYMATH_C19, twins=TWINS_C19) as seen:
            got = function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_19(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:160]}")
        parts_19.append(part)
        raise StopCheck from None
    if seen:
        report_19(False, "", f"({part}) {name} appelle {seen[0]} (de ta librairie ou de la référence ref_pg) : l'énoncé "
                  "demande la correction sans elles, en corrigeant la formule du collègue")
        parts_19.append(part)
        raise StopCheck
    return got


def unchanged_19(arrays, originals):
    return all(np.array_equal(a, np.array(o)) for a, o in zip(arrays, originals))


with checked("MB8.19"):                                       # (1) score_exp_fixed
    got_c19, bad_c19, why_c19 = [], [], ""
    for theta_c19, points_c19 in CASES_C19["scores"]:
        x_c19 = np.array(points_c19)                          # non-dyadic floats: a change in place would show
        value_c19 = as_floats(call_19(1, "score_exp_fixed", score_exp_fixed, x_c19, theta_c19))
        if not unchanged_19([x_c19], [points_c19]):
            why_c19 = "ta fonction modifie le tableau reçu : travaille sur une copie"
            break
        if value_c19 is None or value_c19.shape != x_c19.shape:
            why_c19 = "score_exp_fixed rend un score par point de x, de la forme de x"
            break
        got_c19 += value_c19.tolist()
        bad_c19 += score_exp_bad(np.array(points_c19), theta_c19).tolist()
    if not why_c19 and not passes(CASES_C19["id_scores"], got_c19):
        why_c19 = ("ta correction rend encore ce que rend score_exp_bad" if np.allclose(got_c19, bad_c19, rtol=1e-12)
                   else "pour d'autres taux (ni 1, ni 0,4) et d'autres points, ce n'est pas le score en θ de la loi "
                        "exponentielle de taux θ")
    report_19(not why_c19, "(1) score_exp_fixed : d'autres taux, d'autres points", f"(1) score_exp_fixed : {why_c19}")
    parts_19.append(1)

with checked("MB8.19"):                                       # (2) weights_fixed
    got_c19, bad_c19, why_c19 = [], [], ""
    for lp_c19, lq_c19 in CASES_C19["weights"]:
        left_c19, right_c19 = np.array(lp_c19), np.array(lq_c19)
        value_c19 = as_floats(call_19(2, "weights_fixed", weights_fixed, left_c19, right_c19))
        if not unchanged_19([left_c19, right_c19], [lp_c19, lq_c19]):
            why_c19 = "ta fonction modifie un tableau reçu : travaille sur une copie"
            break
        if value_c19 is None or value_c19.shape != left_c19.shape:
            why_c19 = "weights_fixed rend un poids par tirage, de la forme de log_p"
            break
        got_c19 += value_c19.tolist()
        bad_c19 += weights_bad(np.array(lp_c19), np.array(lq_c19)).tolist()
    if not why_c19 and not passes(CASES_C19["id_weights"], got_c19):
        why_c19 = ("ta correction rend encore ce que rend weights_bad" if np.allclose(got_c19, bad_c19, rtol=1e-12)
                   else "pour d'autres log-densités, ce ne sont pas les poids p(x_i)/q(x_i) des tirages de q")
    report_19(not why_c19, "(2) weights_fixed : d'autres log-densités", f"(2) weights_fixed : {why_c19}")
    parts_19.append(2)

with checked("MB8.19"):                                       # (3) estimate_fixed
    got_c19, bad_c19, why_c19 = [], [], ""
    for f_c19, s_c19 in CASES_C19["batches"]:
        left_c19, right_c19 = np.array(f_c19), np.array(s_c19)
        value_c19 = call_19(3, "estimate_fixed", estimate_fixed, left_c19, right_c19)
        if not unchanged_19([left_c19, right_c19], [f_c19, s_c19]):
            why_c19 = "ta fonction modifie un tableau reçu : travaille sur une copie"
            break
        number_c19 = as_floats(value_c19)
        if number_c19 is None or number_c19.shape != ():
            why_c19 = f"estimate_fixed rend un nombre (l'estimation du batch), reçu {shown(value_c19)}"
            break
        got_c19.append(float(number_c19))
        bad_c19.append(estimate_bad(np.array(f_c19), np.array(s_c19)))
    if not why_c19 and not passes(CASES_C19["id_batches"], got_c19):
        why_c19 = ("ta correction rend encore ce que rend estimate_bad" if np.allclose(got_c19, bad_c19, rtol=1e-12)
                   else "sur d'autres batches (de 2 à 7 tirages, des valeurs f quelconques), ce n'est pas l'estimation "
                        "avec, pour chaque tirage, une baseline calculée sur le batch qui garde l'estimateur sans biais")
    report_19(not why_c19, "(3) estimate_fixed : d'autres batches, de tailles différentes",
              f"(3) estimate_fixed : {why_c19}")
    parts_19.append(3)

with checked("MB8.19"):                                       # (4) strongest_fixed
    why_c19 = ""
    for wins_a_c19, wins_b_c19 in [(TOURNAMENT_A, TOURNAMENT_B), (TOURNAMENT_A, TOURNAMENT_B3)] + CASES_C19["pairs"]:
        try:
            with code_from(MYMATH_C19, twins=TWINS_C19) as seen_c19:
                answer_c19 = strongest_fixed(np.array(wins_a_c19), np.array(wins_b_c19))
        except (NotImplementedError, MymathMissingError):
            raise
        except ValueError:
            answer_c19 = ValueError
        except Exception as exc:  # noqa: BLE001 - the learner's error
            why_c19 = f"ton code lève {type(exc).__name__} : {str(exc)[:160]}"
            break
        if seen_c19:
            why_c19 = (f"appelle {seen_c19[0]} (de ta librairie ou de la référence ref_pg) : l'énoncé demande la "
                       "correction sans elles, en corrigeant la formule du collègue")
            break
        if answer_c19 is not ValueError:
            why_c19 = (f"pour deux tournois sans club commun ni match entre eux, ta fonction désigne encore un club "
                       f"({shown(answer_c19)}) : relis ce que la fiche §7 dit de l'existence des forces et de ce "
                       "qu'elles identifient")
            break
    report_19(not why_c19, "(4) strongest_fixed : les deux tournois, et d'autres paires de tournois",
              f"(4) strongest_fixed : {why_c19}")
    parts_19.append(4)

if len(set(parts_19)) == 4:
    verdict("MB8.19", all(results_19), "les quatre fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chacune des quatre erreurs, pourquoi la vérification du collègue ne la voyait-elle pas, et quelle vérification l'aurait vue ?

### Ex MB8.20 — Un test de propriété pour le gradient exact : sur des MDP tirés au hasard, comparé à des différences finies centrées, et de somme nulle sur les actions de chaque état 🛠️ ★ ⏱️ 25 min
**Objectif :** écrire deux tests de propriété du gradient exact, et voir ce que chacun attrape.  
**Prérequis :** fiche §4 ; M05a §7 (différences finies, choix du pas) ; Ex MB8.14 · **Parcours :** C

Écris deux fonctions qui testent une méthode qu'on leur passe, de la même signature que `policy_gradient_exact(P, r, logits, gamma)` (elle rend `(V, grad)`), et rendent `True` si la propriété tient, `False` sinon :
- `matches_finite_differences_20(method)` tire au hasard plusieurs MDP (de 2 à 5 états, 2 ou 3 actions, des matrices $\mathbf{P}[a]$ stochastiques, des récompenses et des logits quelconques, $\gamma$ parmi $0{,}6$ et $0{,}7$) et compare chaque dérivée rendue à la différence finie centrée des valeurs calculées par `np.linalg.solve`, à $10^{-6}$ près en valeur absolue (plus finement que la tolérance par défaut de `np.allclose`, `rtol=1e-5`, `atol=1e-8`) ;
- `sums_to_zero_20(method)` vérifie, sur des MDP tirés au hasard, que pour tout $s_0$ et tout $s$ la somme des dérivées sur les actions de $s$ est nulle à $10^{-10}$ près.

Choisis le pas de tes différences finies pour que la référence passe : l'erreur d'une différence finie centrée a deux parts, la troncature, de l'ordre de $h^2$, et l'arrondi, de l'ordre de $\frac{\varepsilon}{h}$ (M05a §7).

La vérification lance tes deux tests sur `policy_gradient_exact` de la référence (`ref_pg.policy_gradient_exact`), sur deux autres méthodes justes, puis sur des méthodes fausses cachées, passées sous un nom neutre (un test juge ce que fait la méthode, pas son nom), une au moins par clause de l'énoncé, que chaque test doit refuser quand sa propriété est fausse. Un test exécute la méthode sur ses MDP et compare ses résultats à ce que dit la propriété : il ne lit ni la méthode ni la pile d'appels, et n'appelle pas `policy_gradient_exact` (ni celle de la référence, ni la tienne : la vérification le contrôle). Relève :
- a) les résultats de tes deux tests sur `policy_gradient_exact` de la référence (une liste de 2 booléens, dans l'ordre différences finies, somme nulle).

In [ ]:
def matches_finite_differences_20(method):
    """True when, on several MDPs drawn at random, every derivative of method(P, r, logits, gamma)[1] is within 1e-6
    of the centred finite difference of the values computed by np.linalg.solve; False otherwise."""
    raise NotImplementedError("matches_finite_differences_20 n'est pas encore écrite (04_indices.md si tu bloques)")


def sums_to_zero_20(method):
    """True when, on several MDPs drawn at random, the derivatives of method(P, r, logits, gamma)[1] sum to 0 over the
    actions of each state s, for every s0 and s, within 1e-10; False otherwise."""
    raise NotImplementedError("sums_to_zero_20 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB8_20a = ...
with wbm.attempt("MB8.20"):          # your two tests: ⏳ while they are not written
    answer_MB8_20a = ...  # a) a list of 2 booleans: your two tests on ref_pg.policy_gradient_exact

In [ ]:
for letter, answer in zip("a", [answer_MB8_20a]):
    check_answer(f"MB8.20{letter}", answer)

def load_hidden_c20():
    """The hidden methods (solutions/challenges/mb8_20.py: reading it spoils the exercise), as a module of its own."""
    import importlib.util
    spec = importlib.util.spec_from_file_location("hidden_methods_c20", wbm.repo_root() / "solutions" / "challenges"
                                                  / "mb8_20.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


HIDDEN_C20 = load_hidden_c20()
WRONG_C20 = [   # a hidden wrong method, the test that must refuse it, and the clause of the statement it tries
    (HIDDEN_C20.occupancy, "matches_finite_differences_20", "d'un facteur 1 − γ sur chaque dérivée : ton test compare-t-il "
                                                            "bien les dérivées rendues aux différences finies ?"),
    (HIDDEN_C20.zeros, "matches_finite_differences_20", "partout (elle rend des zéros, de somme nulle) : ton test "
                                                        "compare-t-il bien les dérivées rendues aux différences finies ?"),
    (HIDDEN_C20.transposed, "matches_finite_differences_20", "dès que M n'est pas symétrique : tes MDP sont-ils tirés au "
                                                             "hasard, avec des P[a] quelconques ?"),
    (HIDDEN_C20.uniform, "matches_finite_differences_20", "dès que la politique d'un état n'est pas uniforme : tes "
                                                          "logits sont-ils quelconques (différents d'une action à "
                                                          "l'autre) ?"),
    (HIDDEN_C20.three_actions, "matches_finite_differences_20", "seulement avec 3 actions : tes MDP ont-ils parfois 3 "
                                                                "actions ?"),
    (HIDDEN_C20.four_states, "matches_finite_differences_20", "seulement à partir de 4 états : tes MDP vont-ils jusqu'à "
                                                              "5 états ?"),
    (HIDDEN_C20.mixed_rewards, "matches_finite_differences_20", "dès que les récompenses d'un état diffèrent d'une action "
                                                                "à l'autre : tes récompenses sont-elles quelconques ?"),
    (HIDDEN_C20.relative_8e6, "matches_finite_differences_20", "de 8·10⁻⁶ en valeur relative, ce que np.allclose laisse "
                                                               "passer par défaut (rtol = 1e-5) : ta tolérance est-elle "
                                                               "10⁻⁶ en valeur absolue, sur des dérivées de l'ordre de "
                                                               "1 (des récompenses de l'ordre de 1) ?"),
    (HIDDEN_C20.offset_large, "matches_finite_differences_20", "de 2,5·10⁻⁶ sur les dérivées d'au moins 0,25 en valeur "
                                                               "absolue : ta tolérance est-elle 10⁻⁶ en valeur absolue "
                                                               "(pas plus large, pas relative) ?"),
    (HIDDEN_C20.last_entry, "matches_finite_differences_20", "sur une seule dérivée, de 4·10⁻⁶ : ton test compare-t-il "
                                                             "chaque dérivée rendue, pour tous les s₀, s et a ?"),
    (HIDDEN_C20.values_not_advantages, "sums_to_zero_20", "sur la somme de chaque état (elle prend Q au lieu de "
                                                          "l'avantage) : ton test somme-t-il sur les actions ?"),
    (HIDDEN_C20.offset_sum, "sums_to_zero_20", "de 3·10⁻¹⁰ sur chaque dérivée, ce que np.allclose laisse passer par "
                                               "défaut (atol = 1e-8) : ta tolérance sur la somme est-elle 10⁻¹⁰ ?"),
    (HIDDEN_C20.one_pair, "sums_to_zero_20", "sur la somme d'un seul couple (s₀, s), de 5·10⁻¹⁰ : vérifies-tu tous les s₀ "
                                             "et tous les s, à 10⁻¹⁰ près ?"),
    (HIDDEN_C20.opposite_pairs, "sums_to_zero_20", "sur les sommes de deux couples (s₀, s), de +5·10⁻¹⁰ et −5·10⁻¹⁰, "
                                                   "qui se compensent : ton test juge-t-il chaque somme, pas leur total "
                                                   "ni leur moyenne ?"),
]

with checked("MB8.20"):
    results_20 = []
    report_20 = reporter(results_20, stop=False)
    TESTS_C20 = {"matches_finite_differences_20": matches_finite_differences_20, "sums_to_zero_20": sums_to_zero_20}

    def anonymous_c20(method):
        """method under a neutral name: a test must judge what the method does, not what it is called."""
        def method_c20(P, r, logits, gamma):
            return method(P, r, logits, gamma)
        method_c20.__name__ = method_c20.__qualname__ = "method"
        return method_c20

    def verdict_c20(name, label, method):
        got = call_safely(f"Ex MB8.20 : {name}({label})", TESTS_C20[name], anonymous_c20(method))
        if not isinstance(got, (bool, np.bool_)):
            if isinstance(got, (int, np.integer)):
                what = f"{int(got)} (un entier)"
            elif isinstance(got, (float, np.floating)):
                what = f"{float(got):g} (un nombre)"
            else:
                what = shown(got)
            print(f"❌ Ex MB8.20 : {name}({label}) rend {what}, pas True ou False")
            raise StopCheck
        return bool(got)

    PEEK_C20 = {"__closure__", "cell_contents", "__code__", "__wrapped__", "__defaults__", "__globals__", "__name__",
                "__qualname__", "__doc__", "__module__", "_getframe", "currentframe", "f_back", "f_locals", "f_globals",
                "inspect", "getsource", "get_objects", "WRONG_C20", "TESTS_C20", "HIDDEN_C20", "mb8_20",
                "getattr", "attrgetter", "methodcaller", "vars", "__getattribute__", "__dict__", "eval", "exec",
                "compile", "globals", "locals", "builtins", "__builtins__", "__import__", "importlib", "gc", "ctypes",
                "settrace", "setprofile", "monitoring", "traceback", "walk_stack", "extract_stack"}

    def peeks_c20(function, seen=None):
        """The names of PEEK_C20 that the learner's function (or a function of the notebook it calls) uses, as a name
        or as a string."""
        seen = set() if seen is None else seen
        code = getattr(function, "__code__", None)
        if code is None or code in seen:
            return set()
        seen.add(code)
        found, todo = set(), [code]
        while todo:
            c = todo.pop()
            found |= PEEK_C20 & (set(c.co_names) | {k for k in c.co_consts if isinstance(k, str)})
            for name in c.co_names:
                target = globals().get(name)
                if callable(target) and getattr(target, "__module__", None) == "__main__":
                    found |= peeks_c20(target, seen)
            todo += [k for k in c.co_consts if hasattr(k, "co_code")]
        return found

    peeked_c20 = {name: sorted(peeks_c20(test)) for name, test in TESTS_C20.items()}
    peeked_c20 = {name: names for name, names in peeked_c20.items() if names}
    if peeked_c20:
        name_c20, names_c20 = next(iter(peeked_c20.items()))
        print(f"❌ Ex MB8.20 : {name_c20} lit la méthode ou la pile d'appels ({', '.join(names_c20)}) au lieu de "
              "l'exécuter : un test juge ce que fait la méthode sur tes MDP")
        raise StopCheck

    LIBRARY_FILES_C20 = [REF.policy_gradient.__file__]
    try:
        LIBRARY_FILES_C20.append(mymath.policy_gradient.__file__)          # your own policy_gradient.py too
    except Exception:  # noqa: BLE001 - no module of yours yet: only the reference is watched
        pass

    def watched_c20(name, label, method):
        """The verdict of one test on a method that never calls policy_gradient.py: a call comes from the test."""
        with code_from(LIBRARY_FILES_C20) as seen:
            got = verdict_c20(name, label, method)
        if seen:
            print(f"❌ Ex MB8.20 : {name} appelle policy_gradient_exact ({seen[0]}) : un test compare la méthode à des "
                  "différences finies centrées de valeurs calculées par np.linalg.solve, ou à 0 pour la somme, pas à "
                  "une autre version de la fonction")
            raise StopCheck
        return got

    on_ref_c20 = [verdict_c20(name, "policy_gradient_exact de la référence", ref_pg.policy_gradient_exact)
                  for name in TESTS_C20]
    report_20(passes("MB8.20a", on_ref_c20), "tes deux tests, sur policy_gradient_exact de la référence",
              "tes deux tests ne rendent pas True sur policy_gradient_exact de la référence, qui vérifie les deux "
              "propriétés : relis le choix du pas h (l'arrondi et la troncature, M05a §7) et des tolérances")
    twins_c20 = [watched_c20(name, label, method) for label, method in (("le système dérivé, logit par logit",
                                                                         HIDDEN_C20.per_logit),
                                                                        ("la formule, M par np.linalg.inv",
                                                                         HIDDEN_C20.inverse_formula))
                 for name in TESTS_C20]
    report_20(all(twins_c20), "deux autres méthodes justes reçoivent aussi True",
              "deux autres méthodes justes (le système de Bellman dérivé logit par logit, la formule avec M par "
              "np.linalg.inv ; des arrondis de l'ordre de 10⁻¹⁵) ne reçoivent pas toutes True : ta tolérance est-elle "
              "trop stricte, ou ton pas h mal choisi ?")
    for number_c20, (method_c20, name_c20, clause_c20) in enumerate(WRONG_C20, start=1):
        refused_c20 = not watched_c20(name_c20, f"méthode fausse cachée n° {number_c20}", method_c20)
        report_20(refused_c20, f"{name_c20} refuse la méthode fausse cachée n° {number_c20}",
                  f"{name_c20} accepte la méthode fausse cachée n° {number_c20}, fausse {clause_c20}")
    verdict("MB8.20", all(results_20) and len(results_20) == 2 + len(WRONG_C20), "tes deux tests jugent juste "
            "policy_gradient_exact de la référence et deux autres méthodes justes, et refusent les méthodes fausses "
            "cachées.", "relis les lignes ❌ ci-dessus.")

**Ce qu'il faut regarder** : quel pas $h$ donne le plus petit écart, et pourquoi pas $h = 10^{-12}$ ? Quelle méthode fausse passerait le test de somme nulle seul ? Pourquoi tirer les MDP au hasard, avec des logits non nuls ? Une seule taille de MDP, un seul $\gamma$ ou un oracle analytique écrit à la main passeraient-ils la vérification ? Pourquoi l'énoncé demande-t-il des différences finies centrées ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie D · Défi

Fiche §2 et §3. La dérivée d'une probabilité de dépassement pour une loi inconnue, avec un budget de tirages, vérifiée sur des lois cachées tirées à chaque exécution.

### Ex MB8.21 — Défi : estimer, avec un budget de tirages, la dérivée d'une probabilité de dépassement par rapport au paramètre d'une loi, assez précisément neuf fois sur dix (score, baseline, variable de contrôle) 🏆 ★★★ ⏱️ 40 min
**Objectif :** estimer avec un budget de tirages la dérivée d'une probabilité de dépassement pour une loi qu'on ne connaît pas, et le vérifier sur des lois cachées.  
**Prérequis :** fiche §2 et §3 ; Ex MB8.1, MB8.13, MB8.17 · **Parcours :** C

Écris `derivative_21(draw, c, budget)`, qui estime $J'(\theta) = \frac{\mathrm{d}}{\mathrm{d}\theta}P_\theta(X > c)$ pour une loi $p_\theta$ que tu ne connais pas. Ta fonction peut appeler `draw(k, delta=0.0)` : $k$ tirages, un décalage $\delta$ de $\theta$ (propre à cet exercice). Elle rend deux tableaux de `k` flottants : `k` tirages indépendants $x_i$ de $p_{\theta + \delta}$, puis leurs scores $s_{\theta + \delta}(x_i)$, au paramètre du tirage ; ses appels réunis ne doivent pas dépasser `budget` tirages (au-delà, `draw` rend des `nan`, et le cas échoue). Ni la famille de la loi ni $\theta$ ne sont donnés : la loi est exponentielle, de Poisson, normale ou binomiale, et le seuil $c$ donne une probabilité $P_\theta(X > c)$ qui va de presque 0 à presque 1. Le notebook fournit un cas d'essai, les attentes au téléphone de MB8.1 (`TRIAL_21` : taux $0{,}1$, seuil 18 minutes, budget de 10 000 tirages), et une fonction `draw` pour lui (`trial_draw_21()` en rend une nouvelle, qui compte ses tirages).

**Critère de réussite :** ton estimation est à 10 % près (en valeur relative) de $J'(\theta)$ pour le cas d'essai, et pour au moins 90 % des 120 cas cachés (douze séries de dix), tirés à nouveau à chaque exécution par un processus à part, qui sert les tirages et compte ceux de chaque cas. Ta fonction ne doit ni exécuter le code du module caché du défi (`solutions/challenges/mb8_21.py`), ni lancer un autre processus : la vérification le contrôle. Relève :
- a) ton estimation pour le cas d'essai (un nombre ; la vérification la compare à la dérivée exacte, à 10 % près).

Point de départ : la dérivée par le chemin d'une indicatrice est nulle ; des différences finies entre deux échantillons indépendants, à $\theta + h$ et à $\theta - h$, noient l'écart de deux probabilités voisines dans leur bruit ; l'estimateur score sans baseline est sans biais, mais est-il assez précis dans tous les cas ?

In [ ]:
# The trial case of the challenge: the waits on the phone of MB8.1 (an exponential law of rate 0.1), the threshold 18
# minutes and a budget of 10 000 draws; trial_draw_21() returns a new draw function for it (seed 8125), which counts the
# draws it serves
TRIAL_21 = MappingProxyType({"rate": 0.1, "c": 18.0, "budget": 10_000})


def trial_draw_21(seed=8125):
    """A new function draw(k, delta=0.0) for the trial case: k draws of the exponential law of rate 0.1 + delta and their
    scores at that rate (two float arrays); beyond TRIAL_21["budget"] draws in all, NaNs."""
    rng, used = np.random.default_rng(seed), [0]

    def draw(k, delta=0.0):
        if isinstance(k, (bool, np.bool_)) or not isinstance(k, numbers.Integral) or k < 0:
            raise ValueError(f"draw(k) prend un nombre entier de tirages k >= 0, reçu {k!r}")
        if isinstance(delta, (bool, np.bool_)) or not isinstance(delta, numbers.Real) or not math.isfinite(delta):
            raise ValueError(f"draw(k, delta) prend un décalage delta réel et fini, reçu {delta!r}")
        k, rate = int(k), TRIAL_21["rate"] + float(delta)
        if used[0] + k > TRIAL_21["budget"]:
            used[0] = TRIAL_21["budget"] + 1
            return np.full(k, np.nan), np.full(k, np.nan)
        used[0] += k
        if rate <= 0:
            return np.full(k, np.nan), np.full(k, np.nan)
        x = rng.exponential(1 / rate, k)
        return x, 1 / rate - x
    return draw

In [ ]:
def derivative_21(draw, c, budget):
    """An estimate of J'(θ) = d/dθ P_θ(X > c) for a law p_θ you do not know: draw(k, delta=0.0) returns k draws of
    p_{θ+δ} and their scores at θ + δ (two float arrays); at most budget draws in all."""
    raise NotImplementedError("derivative_21 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB8_21a = ...
with wbm.attempt("MB8.21"):          # your derivative_21: ⏳ while it is not written
    answer_MB8_21a = ...  # a) a number, as computed: your estimate for the trial case (a new trial_draw_21(), TRIAL_21["c"], TRIAL_21["budget"])

In [ ]:
def check_c21():
    """The check of MB8.21, in a function. The hidden laws come from another process (the hidden module run as a
    script): it draws them, serves the draws of each case and counts them, and compares the estimates it receives with
    the exact derivatives only at the end; no hidden law, parameter nor derivative lives in this process while the
    learner's function runs. Each call runs under code_from (the learner's function may not run code of the hidden
    module, nor open it, nor start a process) and under a time limit where the system allows one (it only stops a
    runaway computation)."""
    import json
    import signal
    import subprocess
    import sys
    import threading
    hidden_file = wbm.repo_root() / "solutions" / "challenges" / "mb8_21.py"   # reading it spoils the challenge
    twins = code_keys_of_file(hidden_file)                      # its functions, to recognise a copy run from elsewhere
    results = []
    report = reporter(results, stop=False)
    largest_request = 1_000_000                                 # the most draws asked to the other process at once

    class TimeUp(BaseException):
        """Raised by the alarm: a BaseException, so that a broad except of the learner's code does not hide it."""

    @contextlib.contextmanager
    def time_limit(seconds):
        usable = hasattr(signal, "setitimer") and threading.current_thread() is threading.main_thread()
        if not usable:
            yield
            return

        def ring(signum, frame):
            raise TimeUp

        before = signal.signal(signal.SIGALRM, ring)
        signal.setitimer(signal.ITIMER_REAL, max(seconds, 0.01))
        try:
            yield
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, before)

    def exactly(stream, size):
        data = stream.read(size)
        if len(data) != size:
            raise OSError("échange interrompu")
        return data

    def send(line):
        server.stdin.write((line + "\n").encode())
        server.stdin.flush()

    def draw_of(k, budget):
        """The function draw(n, delta=0.0) of case k: its draws come from the other process, which counts them."""
        used = [0]

        def draw(n, delta=0.0):
            if isinstance(n, (bool, np.bool_)) or not isinstance(n, numbers.Integral) or n < 0:
                raise ValueError(f"draw(k) prend un nombre entier de tirages k >= 0, reçu {n!r}")
            if isinstance(delta, (bool, np.bool_)) or not isinstance(delta, numbers.Real) or not math.isfinite(delta):
                raise ValueError(f"draw(k, delta) prend un décalage delta réel et fini, reçu {delta!r}")
            n = int(n)
            if used[0] + n > budget:
                used[0] = budget + 1
                send(f"O {k}")
                if n > 10 * budget + 10:
                    raise ValueError(f"draw({n}) : {n} tirages, bien plus que le budget de {budget}")
                return np.full(n, np.nan), np.full(n, np.nan)
            used[0] += n
            xs, ss, left = [], [], n
            while True:
                part = min(left, largest_request)
                send(f"D {k} {part} {float(delta)!r}")
                data = np.frombuffer(exactly(server.stdout, 16 * part), dtype="<f8")
                xs.append(data[:part])
                ss.append(data[part:])
                left -= part
                if left <= 0:
                    break
            return np.concatenate(xs).copy(), np.concatenate(ss).copy()
        return draw

    def estimate(label, k, c, budget):
        """derivative_21 on case k, under code_from and a time limit: a float, or None (not a finite number)."""
        try:
            with time_limit(120):
                with code_from([hidden_file], twins=twins, escapes=True) as seen:
                    out = call_safely(label, derivative_21, draw_of(k, budget), float(c), int(budget))
        except TimeUp:
            print(f"❌ {label} : ta fonction n'a pas fini en 120 s : une boucle s'arrête-t-elle ?")
            raise StopCheck from None
        if seen and seen[0].startswith("lance "):
            print(f"❌ {label} : derivative_21 {seen[0].replace(' : ', ' (', 1)}) : l'estimation se calcule dans ta "
                  "fonction ; ce qu'exécute un autre processus échappe à la vérification")
            raise StopCheck
        if seen:
            what = "lit" if seen[0].startswith("ouvre ") else "exécute du code de"
            where = "" if seen[0].startswith("ouvre ") else f" ({seen[0]})"
            print(f"❌ {label} : derivative_21 {what} solutions/challenges/mb8_21.py{where}, la partie cachée du défi, "
                  "ou une copie de son code : la vérification mesurerait sa méthode, pas la tienne")
            raise StopCheck
        value = as_floats(out)
        if value is None or value.shape != ():
            print(f"❌ {label} : derivative_21 rend un nombre, l'estimation de J'(θ), reçu {shown(out)}")
            raise StopCheck
        return float(value) if math.isfinite(float(value)) else None

    server = subprocess.Popen([sys.executable, "-I", str(hidden_file), "--serve"], stdin=subprocess.PIPE,
                              stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)   # this kernel's Python, here or on Colab
    try:
        head = json.loads(server.stdout.readline())
        count, thresholds, budgets = head["count"], head["c"], head["budget"]
        estimates = []
        for k in range(count):
            label = ("Ex MB8.21 : derivative_21 (le cas d'essai, des tirages de l'autre processus)" if k == count - 1
                     else f"Ex MB8.21 : derivative_21 (cas caché {k + 1})")
            value = estimate(label, k, thresholds[k], budgets[k])
            estimates.append(value)
            send(f"R {k} {value!r}" if value is not None else f"R {k} nan")
        given = filled(answer_MB8_21a) and as_floats(answer_MB8_21a) is not None \
            and as_floats(answer_MB8_21a).shape == () and math.isfinite(float(as_floats(answer_MB8_21a)))
        if given:
            send(f"A {float(as_floats(answer_MB8_21a))!r}")
        send("verdict")                                        # only now: the families, the errors, the verdicts
        measured = json.loads(server.stdout.readline())
        cases, answer = measured["cases"], measured["answer"]
        assert len(cases) == count
    except (OSError, ValueError, KeyError, TypeError, AssertionError) as exc:
        print("❌ Ex MB8.21 : la vérification n'a pas pu échanger les tirages et les estimations avec le processus qui "
              f"tire les lois cachées ({type(exc).__name__} ; ta fonction a-t-elle touché à ce processus ?) : exécute à "
              "nouveau la cellule")
        raise StopCheck from None
    finally:                                                    # always, even on an error or an interruption
        if server.poll() is None:
            server.kill()
        server.wait()
        for pipe in (server.stdin, server.stdout):
            try:
                pipe.close()
            except OSError:                                     # a pipe already broken
                pass

    def percent(rel):
        return "non fini" if rel is None else f"{fr(100 * rel, 0)} %"

    notebook, hidden = cases[-1], cases[:-1]
    report(notebook["ok"], "le cas d'essai, avec des tirages de l'autre processus : à 10 % près de J'(θ)",
           "le cas d'essai, avec des tirages de l'autre processus : " + (
               "ta fonction dépasse le budget de 10 000 tirages" if notebook["over_budget"] else
               "ton estimation n'est pas un nombre fini" if estimates[-1] is None else
               "ton estimation est à plus de 10 % de J'(θ)"))
    for number in sorted({case["series"] for case in hidden}):  # what each series of hidden cases measured
        series = [case for case in hidden if case["series"] == number]
        missed = [case for case in series if not case["ok"]]
        kinds = [kind for kind in ("très fréquent", "fréquent", "moyen", "rare") if any(c["kind"] == kind for c in missed)]
        gaps = [case["rel"] for case in missed if case["rel"] is not None and not case["over_budget"]]
        print(f"   série {number} : {len(series) - len(missed)} sur {len(series)} à 10 % près"
              + ("" if not missed else " ; manqués : " + ", ".join(
                  f"« {kind} » ×{sum(c['kind'] == kind for c in missed)}" for kind in kinds)
                 + (f" (écart le plus grand : {percent(max(gaps))})" if gaps else "")
                 + (f" ; {sum(c['over_budget'] for c in missed)} au-delà du budget"
                    if any(c["over_budget"] for c in missed) else "")))
    succeeded = sum(case["ok"] for case in hidden)
    needed = math.ceil(0.9 * len(hidden))
    over = [case for case in hidden if case["over_budget"]]
    hidden_values = [value for value in estimates[:-1] if value is not None]
    if succeeded >= needed:
        why = ""
    elif over:
        why = (f" ; {len(over)} cas dépassent leur budget : les appels de draw, réunis, ne doivent pas demander plus de "
               "budget tirages (au-delà, des nan)")
    elif sum(case["used"] for case in hidden) == 0:
        why = " ; ta fonction n'appelle jamais draw : elle ne connaît des lois cachées que ce que le notebook lui montre"
    elif len(hidden_values) >= 3 and max(hidden_values.count(value) for value in hidden_values) >= 3:
        why = (" ; ta fonction rend la même valeur pour plusieurs cas différents : se sert-elle de nombres du cas "
               "d'essai au lieu des tirages qu'elle reçoit ?")
    else:
        failed = [case for case in hidden if not case["ok"]]
        share = {kind: sum(case["kind"] == kind for case in failed) / len(failed) for kind in
                 ("très fréquent", "fréquent", "moyen", "rare")}
        if share["très fréquent"] + share["fréquent"] >= 0.7:
            why = (" ; la plupart de tes échecs sont des cas « très fréquents » ou « fréquents », où P(X > c) est proche "
                   "de 1 : que vaut alors f(x_i) pour presque tous les tirages, et que devient ton estimateur ?")
        elif share["rare"] >= 0.7:
            why = (" ; la plupart de tes échecs sont des cas « rares », où peu de tirages dépassent c : que vaut alors "
                   "f(x_i) pour presque tous les tirages, et que devient ton estimateur ?")
        else:
            why = (" ; tes échecs touchent toutes les sortes de cas : l'espérance de ton estimation est-elle bien J'(θ), "
                   "et sa dispersion assez petite avec ce budget (fiche §2 et §3) ?")
    report(succeeded >= needed, f"les {len(hidden)} cas cachés : {succeeded} à 10 % près (au moins {needed} demandés)",
           f"les {len(hidden)} cas cachés : {succeeded} à 10 % près, moins que les {needed} demandés" + why)
    report(not over, f"le budget : aucun des {len(hidden)} cas cachés n'en demande plus",
           f"le budget : {len(over)} cas cachés en demandent plus (au-delà, draw rend des nan)")
    if not given:
        print("⏳ Ex MB8.21a : pas encore fait." if not filled(answer_MB8_21a) else
              f"❌ Ex MB8.21a : a) est un nombre fini, ton estimation pour le cas d'essai, reçu {shown(answer_MB8_21a)}")
    else:
        if answer["ok"] and notebook["over_budget"]:            # a) counted by trial_draw_21, in reach of the learner
            print("❌ Ex MB8.21a : ton estimation est à 10 % près, mais, quand l'autre processus compte les tirages du cas "
                  "d'essai, ta fonction dépasse son budget de 10 000 tirages : a) ne compte que dans le budget")
        else:
            print("✅ Ex MB8.21a : ton estimation pour le cas d'essai est à 10 % près de la dérivée exacte."
                  if answer["ok"] else "❌ Ex MB8.21a : ton estimation pour le cas d'essai est à plus de 10 % de la "
                  "dérivée exacte (ta réponse à MB8.1 b) : relis-la, et la méthode de ta fonction")
    verdict("MB8.21", all(results) and len(results) == 3, "critère atteint : le cas d'essai, et au moins neuf cas cachés "
            "sur dix, à 10 % près, sans dépasser les budgets.", "critère pas encore atteint : relis les lignes ❌ "
            "ci-dessus.")


with checked("MB8.21"):
    check_c21()

**Pour réfléchir** (une fois le critère atteint) : sur quelles sortes de cas cachés (les lignes de chaque série) ta première idée a-t-elle échoué, et pourquoi ? Que vaut $f(x_i)$ pour presque tous les tirages d'un cas « très fréquent », d'un cas « rare » ? Ta fonction a-t-elle eu besoin de connaître la famille de la loi ? Les réponses sont dans `05_solutions.md`, section Notebook.

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer la fonction score d'une loi usuelle, et son information de Fisher ?
2. Sais-tu dériver l'espérance d'une indicatrice par la méthode du score, et dire pourquoi la dérivée par le chemin y échoue ?
3. Sais-tu réduire la variance d'un estimateur score par une baseline, et dire laquelle garde l'estimateur sans biais ?
4. Sais-tu calculer le gradient de la valeur d'une politique softmax tabulaire par le système de Bellman ?
5. Sais-tu repondérer des tirages d'une loi pour estimer une moyenne sous une autre, et lire la taille effective ?
6. Sais-tu ajuster les forces d'un modèle de Bradley-Terry, et dire quand elles n'existent pas ?

**La suite** : le chapitre bonus B8 de workbookIA, qui applique ces outils.